# Ghép ảnh panorama: báo cáo giữa kỳ

**CSC14120 Lập trình song song · Nhóm 16 · Phiên bản tuần tự V0**

Báo cáo trình bày cách nhóm hiểu bài toán ghép hai ảnh thành panorama, phương pháp chọn cho sáu stage, phiên bản tuần tự V0, kết quả kiểm chứng và đo thời gian trên Colab, dataset của nhóm và kế hoạch chuyển sang CUDA. Mọi bảng, hình và con số trong phần kết quả được sinh lại mỗi lần chạy notebook, từ mã nguồn trong thư mục `src/` của bài nộp.

## Mục lục

0. [Hướng dẫn chạy notebook](#muc-0)
1. [Thông tin nhóm và kế hoạch làm việc](#muc-1)
2. [Bài toán ghép ảnh panorama](#muc-2)
3. [Phương pháp cho từng stage](#muc-3)
4. [Phiên bản tuần tự V0](#muc-4)
5. [Kết quả và profiling V0](#muc-5)
    - [5.1 Độ chính xác homography trên cặp synthetic](#muc-5-1)
    - [5.2 Panorama của các địa điểm](#muc-5-2)
    - [5.3 Thời gian từng stage](#muc-5-3)
    - [5.4 Hot stage theo độ phân giải](#muc-5-4)
    - [5.5 Ước lượng theo định luật Amdahl](#muc-5-5)
6. [Dataset](#muc-6)
7. [Kế hoạch chuyển sang GPU](#muc-7)
8. [Tài liệu tham khảo và khai báo dùng AI](#muc-8)

Trên Colab có thể mở mục lục ở thanh bên trái (biểu tượng ☰) để chuyển nhanh giữa các mục.

<a name="muc-0"></a>
## 0\. Hướng dẫn chạy notebook

1. Chọn **Runtime → Change runtime type → T4 GPU**. V0 chạy trên CPU của máy Colab, nhưng chương trình được build bằng `nvcc -arch=sm_75` và mọi phiên bản của đồ án được đo trên cùng loại máy này.
2. Đưa thư mục `src/` của bài nộp vào phiên Colab: mở panel **Files** rồi tải `src.zip` vào `/content`, hoặc tạo thư mục `/content/src` và tải các file vào đó. Nếu bỏ qua bước này, cell chuẩn bị sẽ hỏi file `src.zip`.
3. Chọn **Runtime → Run all**. Cell chuẩn bị tự tải [`Group16_support.zip`](https://drive.google.com/file/d/1Jx6o5kUnpvxXVEHfYKD1GBvZMTbSeEGu/view?usp=sharing) từ Drive của nhóm. File này chứa dataset đã chuẩn bị, ảnh nguồn của cặp synthetic, các script Python (sinh cặp synthetic, tính sai số, chạy benchmark) và bộ test, không chứa mã nguồn chương trình. Nếu không tải được, cell sẽ hỏi upload file này.

Phần chạy lâu nhất là benchmark ở mục 5.3; trong lúc benchmark chạy không nên chạy cell khác.

Dataset: [thư mục `dataset/` đã chuẩn bị](https://drive.google.com/drive/folders/1HOvnly1k1QHOsV17qLzSAG4ihszNzCqO?usp=sharing) và [ảnh gốc của các thành viên](https://drive.google.com/drive/folders/127SOMhoY2EwBbkCxGWGPju-EbmG0QiOq?usp=sharing).

Cell kế tiếp chứa các hàm hỗ trợ: chuẩn bị phiên chạy, gọi chương trình `stitch`, đọc kết quả, vẽ bảng và biểu đồ. Trên Colab cell này được thu gọn, bấm **Show code** để xem. Log đầy đủ của mọi lệnh được ghi vào `results/logs/` trong thư mục làm việc của phiên, output của notebook chỉ hiện phần tóm tắt.

In [ ]:
# @title Hàm hỗ trợ của notebook (bấm "Show code" để xem) { display-mode: "form" }
# Authors: Le Ha Thanh Chuong (23120195), Nguyen Ho Anh Tuan (23120185)
# External source: none
# Code in this cell only prepares the Colab session, runs the submitted program and
# formats its output. The stitching program itself is built from the submitted src/.
import base64
import datetime as dt
import html
import io
import json
import math
import os
from pathlib import Path
import platform
import re
import shutil
import stat
import subprocess
import sys
import uuid
import zipfile

import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
import numpy as np
import pandas as pd
from PIL import Image as PILImage
from IPython.display import HTML, display

# ---------------------------------------------------------------- inputs
SRC_UPLOAD_DIR = Path("/content/src")
SRC_ARCHIVE = Path("/content/src.zip")
SUPPORT_ARCHIVE = Path("/content/Group16_support.zip")
SUPPORT_DRIVE_ID = "1Jx6o5kUnpvxXVEHfYKD1GBvZMTbSeEGu"  # Group16_support.zip on the team Drive
RAW_SYNTHETIC_RELATIVE = "raw/tuan/standard1_left.jpg"
EXPECTED_SYNTHETIC_SHA256 = "e9fbe7ecfed6344a36610fe50f2c6019bcd1525f4e217c033ed8ded4f90bf1e8"
NVCC_FLAGS = ["-arch=sm_75", "-std=c++11", "-O2", "--fmad=false",
              "-Xcompiler=-ffp-contract=off,-fno-tree-vectorize,-fno-tree-slp-vectorize"]
# Argument list of scripts/run_benchmark.py; placeholders are filled by run_official_benchmark().
BENCHMARK_ARGS = ["--stitch", "{stitch}", "--dataset", "{dataset}", "--runs", "{runs}", "--out", "{out}"]

SIZES = {"small": (1280, 720), "medium": (1920, 1080), "large": (3840, 2160)}
BENCHMARK_SITES = ("site1", "site2", "site3")
PAIRS = ("site1", "site2", "site3", "challenge")
STAGES = ["S1", "S2", "S3", "S4", "S5", "S6"]
STAGE_LABELS = {"S1": "S1 Tiền xử lý", "S2": "S2 Keypoint", "S3": "S3 Descriptor",
                "S4": "S4 Ghép cặp", "S5": "S5 Homography", "S6": "S6 Warp, blend"}
STAGE_COLORS = dict(zip(STAGES, ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#D55E00", "#56B4E9"]))
SIZE_LABELS = {"small": "small (1280×720)", "medium": "medium (1920×1080)", "large": "large (3840×2160)"}

PROJECT = RESULTS = LOGS = BUILD = STITCH = DATASET = None

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 160, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.titlesize": 11, "legend.frameon": False})


# ---------------------------------------------------------------- commands and logs
class StepFailed(RuntimeError):
    pass


def run(command, log_name, cwd=None, timeout=1800, check=True):
    """Run one command quietly; the full stdout/stderr goes to a log file of this session."""
    command = [str(part) for part in command]
    result = subprocess.run(command, cwd=cwd or PROJECT, capture_output=True, text=True, timeout=timeout)
    if LOGS is not None:
        text = "$ " + " ".join(command) + "\n" + result.stdout + ("\n[stderr]\n" + result.stderr if result.stderr else "")
        with (LOGS / log_name).open("a", encoding="utf-8") as stream:
            stream.write(text + "\n")
    if check and result.returncode != 0:
        tail = "\n".join((result.stdout + "\n" + result.stderr).strip().splitlines()[-30:])
        raise StepFailed(f"Command failed (exit {result.returncode}): {' '.join(command)}\n{tail}")
    return result


def sha256(path):
    import hashlib
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


# ---------------------------------------------------------------- tables
TABLE_CSS = [
    {"selector": "", "props": "border-collapse: collapse; font-size: 14px; margin: 4px 0 14px 0;"},
    {"selector": "caption", "props": "caption-side: top; text-align: left; font-weight: bold; font-size: 14px; padding: 0 0 6px 0;"},
    {"selector": "th", "props": "padding: 6px 12px; border-bottom: 1.5px solid #888; text-align: center;"},
    {"selector": "td", "props": "padding: 5px 12px; border-bottom: 1px solid rgba(128,128,128,0.25);"},
]
HOT_CELL = "background-color: rgba(230, 159, 0, 0.35); font-weight: bold;"


def show_table(frame, caption=None, formats=None, left=(), highlight=None, note=None):
    """Display a DataFrame as a compact styled table (same look in Colab and VS Code)."""
    styler = frame.style.hide(axis="index").set_table_styles(TABLE_CSS)
    if caption:
        styler = styler.set_caption(caption)
    if formats:
        styler = styler.format(formats, na_rep="–")
    right = [column for column in frame.columns if column not in left]
    if left:
        styler = styler.set_properties(subset=list(left), **{"text-align": "left"})
    if right:
        styler = styler.set_properties(subset=right, **{"text-align": "right"})
    if highlight is not None:
        styler = styler.apply(highlight, axis=None)
    display(styler)
    if note:
        display(HTML(f'<div style="font-size:13px; opacity:0.8; margin:-8px 0 12px 0">{html.escape(note)}</div>'))


def show_facts(rows, caption):
    """Two-column key/value table for short summaries."""
    show_table(pd.DataFrame(rows, columns=["Mục", "Kết quả"]), caption, left=("Mục", "Kết quả"))


# ---------------------------------------------------------------- 0. runtime and workspace
def check_runtime():
    gpu = run(["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"], "environment.log", cwd="/").stdout.strip()
    if not re.search(r"\bT4\b", gpu):
        raise StepFailed("This notebook must run on a Colab Tesla T4 runtime (Runtime → Change runtime type).")
    nvcc = run(["nvcc", "--version"], "environment.log", cwd="/").stdout
    release = re.search(r"release ([\d.]+), (V[\d.]+)", nvcc)
    cpu = next((line.split(":", 1)[1].strip() for line in Path("/proc/cpuinfo").read_text().splitlines()
                if line.startswith("model name")), platform.processor())
    return {"gpu": gpu.split(",")[0].strip(), "driver": gpu.split(",")[1].strip(), "cpu": cpu,
            "cuda": f"{release.group(1)} ({release.group(2)})" if release else nvcc.strip().splitlines()[-1],
            "python": platform.python_version()}


def _safe_extract(archive, destination):
    """Extract a ZIP after rejecting path traversal, duplicate targets and symbolic links."""
    destination = Path(destination).resolve()
    with zipfile.ZipFile(archive) as bundle:
        seen = set()
        for entry in bundle.infolist():
            target = (destination / entry.filename.replace("\\", "/")).resolve()
            if (not target.is_relative_to(destination) or target == destination or target in seen
                    or stat.S_ISLNK(entry.external_attr >> 16)):
                raise ValueError("Unsafe or duplicate archive member: " + entry.filename)
            seen.add(target)
        for entry in bundle.infolist():
            target = destination / entry.filename.replace("\\", "/")
            if entry.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with bundle.open(entry) as source, target.open("wb") as output:
                    shutil.copyfileobj(source, output)


def _folders_with(root, marker):
    return [folder for folder in [root, *root.rglob("*")] if folder.is_dir() and (folder / marker).is_file()]


def _request_upload(message):
    from google.colab import files
    print(message)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one ZIP archive.")
    return Path(next(iter(uploaded))).resolve()


def prepare_workspace():
    """Copy the submitted src/ into a fresh folder, then add the support files (dataset, scripts, tests)."""
    global PROJECT, RESULTS, LOGS, BUILD, DATASET
    session = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
    PROJECT = Path("/content") / ("group16-" + session)
    PROJECT.mkdir()
    if (SRC_UPLOAD_DIR / "main.cu").is_file():
        shutil.copytree(SRC_UPLOAD_DIR, PROJECT / "src")
    else:
        archive = SRC_ARCHIVE if SRC_ARCHIVE.is_file() else _request_upload("Tải lên src.zip chứa thư mục src/ của bài nộp.")
        staging = PROJECT / "_src_archive"
        _safe_extract(archive, staging)
        roots = _folders_with(staging, "main.cu")
        if len(roots) != 1:
            raise ValueError("The source archive must contain exactly one folder with main.cu.")
        shutil.move(str(roots[0]), str(PROJECT / "src"))
        shutil.rmtree(staging)
    if not SUPPORT_ARCHIVE.is_file():
        try:
            import gdown
            gdown.download(id=SUPPORT_DRIVE_ID, output=str(SUPPORT_ARCHIVE), quiet=True)
        except Exception as error:  # fall back to a manual upload below
            print("Không tải được file hỗ trợ từ Drive:", error)
    archive = SUPPORT_ARCHIVE if SUPPORT_ARCHIVE.is_file() else _request_upload(
        "Tải lên Group16_support.zip (dataset, scripts, tests, ảnh nguồn synthetic).")
    staging = PROJECT / "_support_archive"
    _safe_extract(archive, staging)
    roots = _folders_with(staging, "dataset/manifest.csv")
    if len(roots) != 1:
        raise ValueError("The support archive must contain exactly one dataset/manifest.csv.")
    if (roots[0] / "src").exists():
        raise ValueError("The support archive must not contain src/; src/ comes only from the submission.")
    for name in ("dataset", "scripts", "tests", "raw"):
        if (roots[0] / name).is_dir():
            shutil.move(str(roots[0] / name), str(PROJECT / name))
    shutil.rmtree(staging)
    os.chdir(PROJECT)
    sys.path.insert(0, str(PROJECT / "scripts"))
    RESULTS = PROJECT / "results"
    LOGS = RESULTS / "logs"
    BUILD = PROJECT / "build"
    for folder in (RESULTS, LOGS, BUILD):
        folder.mkdir(parents=True, exist_ok=True)
    DATASET = PROJECT / "dataset"
    return PROJECT


def verify_inputs():
    """Check the 24 dataset images against the manifest and the synthetic source image hash."""
    from midterm_notebook import validate_dataset
    images = validate_dataset(DATASET)
    raw = PROJECT / RAW_SYNTHETIC_RELATIVE
    if sha256(raw) != EXPECTED_SYNTHETIC_SHA256:
        raise ValueError("The synthetic source image differs from the documented one.")
    sources = sorted(p.name for p in (PROJECT / "src").iterdir() if p.suffix in {".cpp", ".cu", ".h", ".cuh"})
    return {"images": len(images), "sources": sources}


def show_session(environment, inputs):
    show_facts([
        ("GPU", f"{environment['gpu']} (driver {environment['driver']})"),
        ("CPU chạy V0", environment["cpu"]),
        ("CUDA / nvcc", environment["cuda"]),
        ("Python", environment["python"]),
        ("Mã nguồn", f"{len(inputs['sources'])} file .cpp/.cu/.h trong src/ của bài nộp"),
        ("Dataset", f"{inputs['images']}/24 ảnh khớp manifest (kích thước, JPEG RGB, không EXIF, SHA-256)"),
        ("Ảnh nguồn của cặp synthetic", f"{RAW_SYNTHETIC_RELATIVE}, SHA-256 khớp"),
    ], "Môi trường của phiên chạy")


# ---------------------------------------------------------------- 4. build, parameters, tests
def build_program():
    global STITCH
    sources = sorted(str(p.relative_to(PROJECT)) for p in (PROJECT / "src").glob("*.cpp"))
    STITCH = BUILD / "stitch"
    command = ["nvcc", *NVCC_FLAGS, "src/main.cu", *sources, "-o", STITCH]
    run(command, "build.log")
    shown = ["nvcc", *NVCC_FLAGS, "src/main.cu", "src/*.cpp", "-o", "build/stitch"]
    display(HTML('<div style="font-size:14px">Build thành công từ <code>src/main.cu</code> và '
                 f'{len(sources)} file <code>.cpp</code>:</div><pre style="font-size:13px">'
                 + html.escape(" \\\n    ".join([" ".join(shown[:6]), " ".join(shown[6:])])) + "</pre>"))
    return STITCH


PARAMETERS = [  # (stage, meaning, file, constant name)
    ("S1", "σ của Gaussian", "v0_config.h", "kV0GaussianSigma"),
    ("S1", "Bán kính kernel Gaussian", "v0_config.h", "kV0GaussianRadius"),
    ("S2", "Số điểm trên vòng FAST", "fast_corner.h", "kFastCircleSize"),
    ("S2", "Số điểm liên tiếp cần sáng/tối hơn", "fast_corner.h", "kFastArcLength"),
    ("S2", "Bán kính vòng FAST", "fast_corner.h", "kFastRadius"),
    ("S2", "Ngưỡng FAST t (thang [0, 1])", "v0_config.h", "kV0FastThreshold"),
    ("S2", "Số keypoint tối đa mỗi ảnh", "v0_config.h", "kV0MaxKeypoints"),
    ("S2, S3", "Lề ảnh không lấy keypoint (pixel)", "feature_border.h", "kFeatureBorder"),
    ("S3", "Số bit của descriptor BRIEF", "brief_pattern.h", "kBriefTests"),
    ("S3", "Bán kính patch (patch 31×31)", "brief_pattern.h", "kBriefPatchRadius"),
    ("S3", "σ làm mượt trước khi so sánh", "brief_pattern.h", "kBriefSmoothSigma"),
    ("S3", "Bán kính kernel làm mượt", "brief_pattern.h", "kBriefSmoothRadius"),
    ("S4", "Ngưỡng ratio test", "feature_support.h", "kRatio"),
    ("S5", "Số lần thử RANSAC tối đa", "feature_support.h", "kMaxIterations"),
    ("S5", "Seed ngẫu nhiên", "feature_support.h", "kSeed"),
    ("S5", "Độ tin cậy để dừng sớm", "feature_support.h", "kConfidence"),
    ("S5", "Ngưỡng inlier ở 1920×1080 (pixel)", "feature_support.h", "kReferenceThreshold"),
    ("S6", "Cạnh canvas tối đa (pixel)", "v0_warp_blend.cpp", "kMaxCanvasDimension"),
    ("S6", "Diện tích canvas tối đa (pixel)", "v0_config.h", "kV0MaxCanvasPixels"),
]


def _constant(file_name, name):
    text = (PROJECT / "src" / file_name).read_text(encoding="utf-8")
    match = re.search(r"constexpr\s+[\w:]+\s+" + name + r"\s*=\s*([^;]+);", text)
    if not match:
        raise ValueError(f"{name} not found in src/{file_name}")
    value = match.group(1).strip()
    value = re.sub(r"(?<=\d)(ULL|ull|u|U|f|F)$", "", value)
    return value


def show_parameters():
    """Main parameters read from the constexpr constants of the submitted src/."""
    rows = [(stage, meaning, _constant(file_name, name), f"{file_name}: {name}")
            for stage, meaning, file_name, name in PARAMETERS]
    preprocess = (PROJECT / "src/v0_preprocess.cpp").read_text(encoding="utf-8")
    weights = re.search(r"\(([\d.]+)f \* r \+ ([\d.]+)f \* g \+ ([\d.]+)f \* b\) / 255", preprocess)
    rows.insert(0, ("S1", "Trọng số R, G, B của ảnh xám", ", ".join(weights.groups()), "v0_preprocess.cpp"))
    warp = (PROJECT / "src/v0_warp_blend.cpp").read_text(encoding="utf-8")
    blend = re.search(r"has_a \? ([\d.]+) \* \(destination\[c\] \+ color\[c\]\)", warp)
    rows.append(("S6", "Trọng số trộn vùng chồng lấn", f"{blend.group(1)} cho mỗi ảnh", "v0_warp_blend.cpp"))
    frame = pd.DataFrame(rows, columns=["Stage", "Tham số", "Giá trị", "Nơi khai báo trong src/"])
    show_table(frame, "Tham số chính, đọc trực tiếp từ mã nguồn bài nộp", left=tuple(frame.columns))


def parse_debug(log):
    """Diagnostics printed by stitch --debug (outside the timed region)."""
    info = {}
    if (m := re.search(r"\[debug\] config: (.*)", log)):
        info["config"] = dict(item.split("=", 1) for item in m.group(1).split())
    if (m := re.search(r"S2 keypoints A=(\d+) B=(\d+); kept by S3 A=(\d+) B=(\d+)", log)):
        info["keypoints_a"], info["keypoints_b"], info["kept_a"], info["kept_b"] = map(int, m.groups())
    if (m := re.search(r"S4 matches=(\d+)", log)):
        info["matches"] = int(m.group(1))
    if (m := re.search(r"\[debug\] S5 (.*)", log)):
        info["s5"] = dict(item.split("=", 1) for item in m.group(1).split())
        info["inliers"] = int(info["s5"]["final_inliers"])
        info["attempts"] = int(info["s5"]["attempts"])
    if (m := re.search(r"S6 canvas=(\d+)x(\d+) offset=\((-?\d+),(-?\d+)\)", log)):
        info["canvas"] = (int(m.group(1)), int(m.group(2)))
        info["offset"] = (int(m.group(3)), int(m.group(4)))
    if (m := re.search(r"H\(B->A\)=(.*)", log)):
        info["H"] = np.array([float(v) for v in m.group(1).split()]).reshape(3, 3)
    return info


def stitch(left, right, output, runs=1, log_name="stitch.log"):
    from midterm_notebook import parse_stitch_csv
    result = run([STITCH, left, right, output, "--version", "0", "--runs", runs, "--debug"], log_name)
    timing = parse_stitch_csv(result.stdout, runs=int(runs))
    return {"csv": result.stdout, "log": result.stderr, "timing": timing, **parse_debug(result.stderr)}


def demo_run():
    """One run per resolution on site1: shows the CSV the program prints and the per-size configuration."""
    rows = []
    for size in SIZES:
        out = stitch(DATASET / "site1" / f"{size}_left.jpg", DATASET / "site1" / f"{size}_right.jpg",
                     RESULTS / f"demo_{size}.png", runs=1, log_name="demo.log")
        config = out["config"]
        rows.append({"Kích thước": SIZE_LABELS[size], "gaussian_sigma": config["gaussian_sigma"],
                     "fast_threshold": float(config["fast_threshold"]), "max_keypoints": config["max_keypoints"],
                     "match_ratio": float(config["match_ratio_threshold"]),
                     "cross_check": config["match_cross_check"], "ransac_iters": config["ransac_iters"],
                     "ransac_threshold (px)": config["ransac_threshold"], "seed": config["seed"]})
        if size == "small":
            sample = out["csv"]
    display(HTML('<div style="font-size:14px">Output CSV của <code>stitch dataset/site1/small_left.jpg '
                 'dataset/site1/small_right.jpg out.png --version 0 --runs 1</code>:</div>'
                 f'<pre style="font-size:13px">{html.escape(sample)}</pre>'))
    show_table(pd.DataFrame(rows), "Cấu hình chương trình in ra khi chạy (--debug), theo kích thước ảnh",
               formats={"fast_threshold": "{:.2f}", "match_ratio": "{:.2f}"}, left=("Kích thước",))


TEST_SCOPE = {
    "S4": "S4: Hamming, top-2, ratio test đúng ranh giới, thứ tự output",
    "DLT": "S5: DLT chuẩn hóa và SVD trên điểm có H biết trước",
    "sampling": "S5: chọn mẫu theo seed, điều kiện dừng thích nghi",
    "MSAC": "S5: hàm điểm MSAC",
    "S5": "S5: toàn bộ RANSAC, refit, H bàn giao",
    "allocation": "S4, S5: dọn bộ nhớ khi cấp phát lỗi",
    "S6": "S6: canvas, warp ngược, bilinear, trộn, xử lý lỗi",
    "test_v0_preprocess": "S1: ảnh xám, blur tách so với Gaussian 2D, biên, lỗi",
    "test_v0_detect": "S2: FAST-9 so với cài đặt vét cạn, NMS, top-N",
    "test_v0_describe": "S3: bit BRIEF, descriptor sau phép dịch, loại keypoint ở biên",
    "test_v0_detect_describe": "S1–S5 trên cặp ảnh dịch biết trước",
    "test_v0_config": "Cấu hình V0 theo kích thước ảnh",
    "test_v0_pipeline": "S1–S6 trên cặp ảnh dịch biết trước, so cả pixel panorama",
}
EXTRA_TESTS = ["test_v0_preprocess", "test_v0_detect", "test_v0_describe",
               "test_v0_detect_describe", "test_v0_config", "test_v0_pipeline"]
PYTHON_TESTS = {
    "test_synthetic_scripts.py": "Sinh cặp synthetic và tính sai số H",
    "test_dataset_scripts.py": "Kiểm tra cặp ảnh và chuẩn bị dataset",
    "test_run_benchmark.py": "Script chạy benchmark",
    "test_plot_benchmark.py": "Script tổng hợp bảng timing",
    "test_midterm_notebook.py": "Đọc CSV timing, kiểm tra dataset",
}


def run_tests():
    """Unit and integration tests of the submitted program; full logs are kept in results/logs/tests.log."""
    found = {}
    runner = run([sys.executable, "tests/run_v0_features.py", "--compiler", "nvcc", "--require-t4"], "tests.log")
    for line in runner.stdout.splitlines():
        m = re.match(r"PASS (S4|DLT|sampling|MSAC|S5|allocation): (\d+) checks", line)
        if m:
            found[m.group(1)] = f"{m.group(2)} check"
        m = re.match(r"PASS: (\d+) assertions for S6", line)
        if m:
            found["S6"] = f"{m.group(1)} assertion"
    if "PASS suites: 7" not in runner.stdout:
        raise StepFailed("tests/run_v0_features.py did not report all suites passing")
    sources = sorted(str(p.relative_to(PROJECT)) for p in (PROJECT / "src").glob("*.cpp"))
    for name in EXTRA_TESTS:
        binary = BUILD / name
        run(["nvcc", *NVCC_FLAGS, "-Isrc", *sources, f"tests/{name}.cpp", "-o", binary], "tests.log")
        output = run([binary], "tests.log").stdout
        m = re.search(r"PASS: (\d+) assertions", output)
        if not m:
            raise StepFailed("Test did not report PASS: " + name)
        found[name] = f"{m.group(1)} assertion"
    order = ["test_v0_preprocess", "test_v0_detect", "test_v0_describe", "S4", "DLT", "sampling", "MSAC",
             "S5", "allocation", "S6", "test_v0_config", "test_v0_detect_describe", "test_v0_pipeline"]
    missing = [key for key in order if key not in found]
    if missing:
        raise StepFailed("Missing test results: " + ", ".join(missing))
    rows = [(TEST_SCOPE[key], found[key], "Đạt") for key in order]
    for file_name, scope in PYTHON_TESTS.items():
        result = run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", file_name], "tests.log")
        text = result.stdout + result.stderr
        count = re.search(r"Ran (\d+) tests?", text)
        skipped = re.search(r"skipped=(\d+)", text)
        if not count or not re.search(r"^OK", text, re.M):
            raise StepFailed("Python test failed: " + file_name)
        label = f"{count.group(1)} test" + (f", {skipped.group(1)} bỏ qua" if skipped else "")
        rows.append(("Công cụ Python: " + scope, label, "Đạt"))
    frame = pd.DataFrame(rows, columns=["Phạm vi kiểm thử", "Số kiểm tra", "Kết quả"])
    show_table(frame, "Kết quả kiểm thử trên Colab T4", left=("Phạm vi kiểm thử",),
               note="Log đầy đủ của từng bộ test nằm trong results/logs/tests.log của phiên chạy.")


def run_cli_cases():
    left, right = DATASET / "site1/small_left.jpg", DATASET / "site1/small_right.jpg"
    output = RESULTS / "cli_output.png"
    bad = RESULTS / "not_an_image.jpg"
    bad.write_text("invalid image bytes", encoding="utf-8")
    cases = [
        ("Thiếu tham số", [], "InvalidArgument"),
        ("Cờ không tồn tại", [left, right, output, "--version", "0", "--unknown"], "UnknownOption"),
        ("Phiên bản chưa có (--version 1)", [left, right, output, "--version", "1"], "UnsupportedVersion"),
        ("--runs 0", [left, right, output, "--version", "0", "--runs", "0"], "InvalidArgument"),
        ("File ảnh không tồn tại", [RESULTS / "missing.jpg", right, output, "--version", "0"], "InputNotFound"),
        ("File không phải ảnh", [bad, right, output, "--version", "0"], "ImageReadFailed"),
    ]
    rows = []
    for label, arguments, expected in cases:
        result = run([STITCH, *arguments], "cli.log", check=False)
        if result.returncode == 0 or f"[{expected}]" not in result.stderr:
            raise StepFailed("Unexpected CLI behavior: " + label)
        rows.append((label, expected, result.returncode))
    show_table(pd.DataFrame(rows, columns=["Trường hợp", "Mã lỗi in ra", "Exit code"]),
               "Xử lý lỗi dòng lệnh", left=("Trường hợp", "Mã lỗi in ra"))


# ---------------------------------------------------------------- 5.1 synthetic accuracy
def _normalized(matrix):
    matrix = np.asarray(matrix, dtype=float).reshape(3, 3)
    return matrix / matrix[2, 2]


def _thumb(path, width):
    with PILImage.open(path) as image:
        image = image.convert("RGB")
        image.thumbnail((width, width))
        return np.asarray(image)


def run_synthetic():
    from homography_common import load_truth
    from homography_error import evaluate
    folder = RESULTS / "synthetic"
    run([sys.executable, "scripts/make_synthetic_pair.py", "--source", PROJECT / RAW_SYNTHETIC_RELATIVE,
         "--out", folder], "synthetic.log")
    rows, matrices = [], {}
    for case in ("translation", "projective"):
        for size in SIZES:
            prefix = folder / case / size
            out = stitch(f"{prefix}_left.png", f"{prefix}_right.png", f"{prefix}_pano.png", log_name="synthetic.log")
            Path(f"{prefix}.log").write_text(out["log"], encoding="utf-8")
            truth = load_truth(f"{prefix}_truth.json")
            metrics = evaluate(truth, out["H"])
            passed = metrics["rmse_px"] <= 0.5 and metrics["max_px"] <= 1.0
            rows.append({"Trường hợp": case, "Kích thước": size, "Match S4": out["matches"],
                         "Inlier S5": out["inliers"], "Điểm kiểm tra": f"{metrics['points']}/{metrics['grid_points']}",
                         "RMSE (px)": metrics["rmse_px"], "Max (px)": metrics["max_px"],
                         "NRMSE": metrics["nrmse"], "Đạt ngưỡng": "Đạt" if passed else "Không đạt"})
            matrices[(case, size)] = (np.asarray(truth["H_gt"], dtype=float).reshape(3, 3), out["H"])
    frame = pd.DataFrame(rows)
    frame.to_csv(RESULTS / "synthetic_accuracy.csv", index=False)
    show_table(frame, "Sai số homography trên 6 cặp synthetic (ngưỡng: RMSE ≤ 0.5 px và Max ≤ 1 px)",
               formats={"RMSE (px)": "{:.6f}", "Max (px)": "{:.6f}", "NRMSE": "{:.3e}"},
               left=("Trường hợp", "Kích thước"))
    worst = frame.loc[frame["Max (px)"].idxmax()]
    display(HTML(f'<div style="font-size:14px">RMSE lớn nhất: <b>{frame["RMSE (px)"].max():.6f} px</b>; '
                 f'Max lớn nhất: <b>{frame["Max (px)"].max():.6f} px</b> ({worst["Trường hợp"]}, {worst["Kích thước"]}).</div>'))
    if (frame["Đạt ngưỡng"] != "Đạt").any():
        raise StepFailed("Synthetic accuracy failed; the V0 baseline is not accepted.")
    return matrices


def show_synthetic_example(matrices, case="projective", size="medium"):
    folder = RESULTS / "synthetic" / case
    figure, axes = plt.subplots(1, 3, figsize=(12, 2.9), gridspec_kw={"width_ratios": [1, 1, 1.75]})
    for axis, name, title in zip(axes, ("left", "right", "pano"),
                                 ("Ảnh A (cắt từ ảnh thật)", "Ảnh B (sinh từ A bằng H_gt)", "Panorama V0 từ A và B")):
        axis.imshow(_thumb(folder / f"{size}_{name}.png", 900))
        axis.set_title(title, fontsize=10)
        axis.axis("off")
    figure.tight_layout()
    plt.show()
    truth, estimate = (_normalized(m) for m in matrices[(case, size)])
    cells = lambda m: "".join("<tr>" + "".join(f"<td style='padding:2px 10px;text-align:right'>{v:.6g}</td>" for v in row) + "</tr>" for row in m)
    display(HTML(f'<div style="font-size:14px">Cặp {case}, {size}: H thật và H do V0 ước lượng (cùng chuẩn hóa để h₃₃ = 1)</div>'
                 '<div style="display:flex; gap:40px; font-size:13px; font-family:monospace">'
                 f'<div><b>H_gt</b><table>{cells(truth)}</table></div>'
                 f'<div><b>H_est</b><table>{cells(estimate)}</table></div></div>'))


def check_repeatability():
    prefix = RESULTS / "synthetic" / "projective" / "small"
    first = parse_debug(Path(f"{prefix}.log").read_text(encoding="utf-8"))
    second = stitch(f"{prefix}_left.png", f"{prefix}_right.png", f"{prefix}_repeat.png", log_name="repeat.log")
    with PILImage.open(f"{prefix}_pano.png") as a, PILImage.open(f"{prefix}_repeat.png") as b:
        same_pixels = a.size == b.size and np.array_equal(np.asarray(a), np.asarray(b))
    checks = [("H (float32) giống nhau", bool(np.array_equal(first["H"], second["H"]))),
              ("Số match giống nhau", first["matches"] == second["matches"]),
              ("Mọi pixel panorama giống nhau", bool(same_pixels))]
    show_facts([(name, "Đúng" if ok else "Sai") for name, ok in checks],
               "Chạy lại cặp projective, small với cùng binary, input và cấu hình")
    if not all(ok for _, ok in checks):
        raise StepFailed("V0 output changed between two identical runs.")


# ---------------------------------------------------------------- 5.2 panoramas and crops
def _image_tag(path, width=None, max_width="100%"):
    with PILImage.open(path) as image:
        image = image.convert("RGB")
        if width:
            image.thumbnail((width, width))
        buffer = io.BytesIO()
        image.save(buffer, format="JPEG" if width else "PNG", quality=88)
    mime = "jpeg" if width else "png"
    data = base64.b64encode(buffer.getvalue()).decode()
    return (f'<img src="data:image/{mime};base64,{data}" style="max-width:{max_width}; '
            'border:1px solid #999; display:block">')


PANORAMA_SIZE = "medium"


def run_panoramas():
    folder = RESULTS / "panoramas"
    folder.mkdir(exist_ok=True)
    rows, info = [], {}
    for pair in PAIRS:
        prefix = folder / f"{pair}_{PANORAMA_SIZE}"
        out = stitch(DATASET / pair / f"{PANORAMA_SIZE}_left.jpg", DATASET / pair / f"{PANORAMA_SIZE}_right.jpg",
                     f"{prefix}.png", log_name="panoramas.log")
        Path(f"{prefix}.log").write_text(out["log"], encoding="utf-8")
        info[pair] = out
        rows.append({"Cặp": pair, "Keypoint A / B": f"{out['keypoints_a']} / {out['keypoints_b']}",
                     "Match S4": out["matches"], "Inlier S5": out["inliers"],
                     "Tỷ lệ inlier": out["inliers"] / out["matches"], "Lần thử RANSAC": out["attempts"],
                     "Canvas": f"{out['canvas'][0]}×{out['canvas'][1]}"})
    show_table(pd.DataFrame(rows), f"Kết quả chạy V0 trên bốn cặp ở kích thước {SIZE_LABELS[PANORAMA_SIZE]}",
               formats={"Tỷ lệ inlier": "{:.0%}"}, left=("Cặp", "Keypoint A / B", "Canvas"))
    return info


def show_panorama(pair, caption):
    path = RESULTS / "panoramas" / f"{pair}_{PANORAMA_SIZE}.png"
    with PILImage.open(path) as image:
        size = image.size
    display(HTML(f'<figure style="margin:6px 0 18px 0">{_image_tag(path, width=1400)}'
                 f'<figcaption style="font-size:13px; margin-top:4px">{html.escape(caption)} '
                 f'(canvas {size[0]}×{size[1]}, ảnh thu nhỏ để xem tổng thể)</figcaption></figure>'))


def _overlap_crops(pair):
    from midterm_notebook import overlap_box
    width, height = SIZES[PANORAMA_SIZE]
    log = (RESULTS / "panoramas" / f"{pair}_{PANORAMA_SIZE}.log").read_text(encoding="utf-8")
    x0, y0, x1, y1 = overlap_box(log, width, height, width, height)
    boxes = []
    for fraction in (0.2, 0.5, 0.8):
        crop_w, crop_h = min(512, x1 - x0), min(512, y1 - y0)
        cx = max(x0, min((x0 + x1) // 2 - crop_w // 2, x1 - crop_w))
        cy = max(y0, min(int(y0 + fraction * (y1 - y0)) - crop_h // 2, y1 - crop_h))
        boxes.append((cx, cy, cx + crop_w, cy + crop_h))
    return boxes


def show_crops(pair):
    """Three 512×512 crops inside the overlap, shown at native resolution (1 image pixel = 1 screen pixel)."""
    path = RESULTS / "panoramas" / f"{pair}_{PANORAMA_SIZE}.png"
    tiles = []
    with PILImage.open(path) as image:
        for index, box in enumerate(_overlap_crops(pair), 1):
            crop_path = RESULTS / "panoramas" / f"{pair}_crop_{index}.png"
            image.crop(box).save(crop_path)
            tiles.append(f'<figure style="margin:0">{_image_tag(crop_path, max_width="none")}'
                         f'<figcaption style="font-size:12px">{pair}, vùng {index}: x {box[0]}–{box[2]}, '
                         f'y {box[1]}–{box[3]}</figcaption></figure>')
    display(HTML('<div style="display:flex; flex-wrap:wrap; gap:10px; margin-bottom:16px">' + "".join(tiles) + "</div>"))


# ---------------------------------------------------------------- 5.3 benchmark
def run_official_benchmark(runs=5):
    from midterm_notebook import collect_benchmark_csv
    out = RESULTS / "benchmark"
    values = {"stitch": str(STITCH), "dataset": str(DATASET), "runs": str(runs), "out": str(out)}
    run([sys.executable, "scripts/run_benchmark.py", *[a.format(**values) for a in BENCHMARK_ARGS]],
        "benchmark.log", timeout=7200)
    average, minimum, measured = (pd.DataFrame(x) for x in collect_benchmark_csv(out, runs))
    for frame in (average, minimum, measured):
        frame.rename(columns={f"{s}_ms": s for s in STAGES} | {"Total_ms": "Total"}, inplace=True)
    details = []
    for site in BENCHMARK_SITES:
        for size in SIZES:
            info = parse_debug((out / site / f"{size}.log").read_text(encoding="utf-8"))
            details.append({"site": site, "resolution": size, "kept_a": info["kept_a"], "kept_b": info["kept_b"],
                            "matches": info["matches"], "canvas_mpx": info["canvas"][0] * info["canvas"][1] / 1e6})
    average.to_csv(RESULTS / "timing_average.csv", index=False)
    measured.to_csv(RESULTS / "timing_runs.csv", index=False)
    return average, minimum, measured, pd.DataFrame(details)


def _hot_stage(frame):
    styles = pd.DataFrame("", index=frame.index, columns=frame.columns)
    stage_columns = [c for c in frame.columns if c in STAGES]
    for row in frame.index:
        styles.loc[row, frame.loc[row, stage_columns].astype(float).idxmax()] = HOT_CELL
    return styles


def show_timing_tables(average, measured, details):
    table = average.copy()
    table.insert(0, "Cấu hình", table["site"] + ", " + table["resolution"])
    table = table[["Cấu hình", *STAGES, "Total"]]
    show_table(table, "Thời gian trung bình của 5 lần đo (ms); ô tô màu là hot stage của từng cấu hình",
               formats={c: "{:.1f}" for c in [*STAGES, "Total"]}, left=("Cấu hình",), highlight=_hot_stage)

    spread = measured.groupby(["site", "resolution"], sort=False)["Total"].agg(["mean", "min", "max", "std"]).reset_index()
    spread["cv"] = spread["std"] / spread["mean"]
    spread.insert(0, "Cấu hình", spread["site"] + ", " + spread["resolution"])
    spread = spread.drop(columns=["site", "resolution"]).rename(columns={
        "mean": "Trung bình (ms)", "min": "Nhỏ nhất (ms)", "max": "Lớn nhất (ms)", "std": "Độ lệch chuẩn (ms)", "cv": "Hệ số biến thiên"})
    show_table(spread, "Độ dao động của Total qua 5 lần đo", left=("Cấu hình",),
               formats={"Trung bình (ms)": "{:.1f}", "Nhỏ nhất (ms)": "{:.1f}", "Lớn nhất (ms)": "{:.1f}",
                        "Độ lệch chuẩn (ms)": "{:.1f}", "Hệ số biến thiên": "{:.1%}"})

    share = average.copy()
    for stage in STAGES:
        share[stage] = 100 * average[stage] / average["Total"]
    mean_share = share.groupby("resolution")[STAGES].mean().loc[list(SIZES)].reset_index()
    mean_share.insert(0, "Kích thước", mean_share["resolution"].map(SIZE_LABELS))
    mean_share = mean_share.drop(columns="resolution")
    show_table(mean_share, "Tỷ trọng trung bình của từng stage trên ba site (% của Total)",
               formats={s: "{:.1f}" for s in STAGES}, left=("Kích thước",), highlight=_hot_stage)

    work = details.copy()
    work.insert(0, "Cấu hình", work["site"] + ", " + work["resolution"])
    work["Descriptor A / B"] = work["kept_a"].astype(str) + " / " + work["kept_b"].astype(str)
    work["S4 (ms)"] = average["S4"].values
    work["Canvas (Mpx)"] = work["canvas_mpx"]
    work["S6 (ms)"] = average["S6"].values
    work = work[["Cấu hình", "Descriptor A / B", "S4 (ms)", "Canvas (Mpx)", "S6 (ms)"]]
    show_table(work, "S4 phụ thuộc số descriptor, S6 phụ thuộc diện tích canvas", left=("Cấu hình", "Descriptor A / B"),
               formats={"S4 (ms)": "{:.1f}", "Canvas (Mpx)": "{:.2f}", "S6 (ms)": "{:.1f}"})
    return share


def plot_stage_share(average):
    order = [(site, size) for size in SIZES for site in BENCHMARK_SITES]
    frame = average.set_index(["site", "resolution"]).loc[order]
    labels = [f"{site} · {size}" for site, size in order]
    y = np.arange(len(order))[::-1] + np.repeat([0.8, 0.4, 0.0], 3)
    figure, axis = plt.subplots(figsize=(11, 5.2))
    left = np.zeros(len(order))
    for stage in STAGES:
        values = 100 * frame[stage].values / frame["Total"].values
        axis.barh(y, values, left=left, color=STAGE_COLORS[stage], label=STAGE_LABELS[stage], height=0.8,
                  edgecolor="white", linewidth=0.6)
        for yi, li, vi in zip(y, left, values):
            if vi >= 6:
                axis.text(li + vi / 2, yi, f"{vi:.0f}%", ha="center", va="center", fontsize=8.5,
                          color="white" if stage in ("S1", "S4", "S5") else "black")
        left += values
    for yi, total in zip(y, frame["Total"].values):
        axis.text(101, yi, f"{total:.0f} ms", va="center", fontsize=9)
    axis.set_yticks(y, labels)
    axis.set_xlim(0, 112)
    axis.set_xticks(range(0, 101, 20), [f"{v}%" for v in range(0, 101, 20)])
    axis.set_xlabel("Tỷ trọng trong Total (%) · số bên phải là Total trung bình")
    axis.set_title("Phân bổ thời gian V0 theo stage", loc="left", pad=34)
    axis.legend(ncol=6, loc="lower left", bbox_to_anchor=(0, 1.04), fontsize=9, handlelength=1.2, columnspacing=1.2)
    axis.tick_params(axis="y", length=0)
    axis.spines["left"].set_visible(False)
    figure.tight_layout()
    figure.savefig(RESULTS / "stage_share.png", bbox_inches="tight")
    plt.show()


def plot_stage_scaling(average):
    pixels = {size: w * h / 1e6 for size, (w, h) in SIZES.items()}
    figure, axis = plt.subplots(figsize=(9.5, 5))
    x = np.array([pixels[size] for size in SIZES])
    for stage in STAGES:
        table = average.pivot(index="resolution", columns="site", values=stage).loc[list(SIZES)]
        mean, low, high = table.mean(axis=1).values, table.min(axis=1).values, table.max(axis=1).values
        axis.fill_between(x, low, high, color=STAGE_COLORS[stage], alpha=0.15, linewidth=0)
        axis.plot(x, mean, marker="o", color=STAGE_COLORS[stage], linewidth=2)
        axis.annotate(STAGE_LABELS[stage], (x[-1], mean[-1]), xytext=(8, 0), textcoords="offset points",
                      va="center", fontsize=9, color=STAGE_COLORS[stage], fontweight="bold")
    axis.set_xscale("log")
    axis.set_yscale("log")
    axis.set_xticks(x, [f"{size}\n{pixels[size]:.2f} Mpx" for size in SIZES])
    axis.minorticks_off()
    axis.set_xlim(x[0] / 1.25, x[-1] * 2.3)
    axis.set_ylabel("Thời gian trung bình mỗi cặp ảnh (ms, thang log)")
    axis.grid(axis="y", alpha=0.3)
    axis.set_title("Thời gian từng stage khi tăng độ phân giải", loc="left", pad=20)
    axis.text(0, 1.01, "Đường: trung bình ba site. Vùng mờ: từ site nhỏ nhất đến site lớn nhất.", transform=axis.transAxes, fontsize=9, color="#666")
    figure.tight_layout()
    figure.savefig(RESULTS / "stage_scaling.png", bbox_inches="tight")
    plt.show()


def amdahl_table(average, stages):
    columns = [f"S{s}" for s in stages]
    frame = average[["site", "resolution"]].copy()
    frame["p"] = average[columns].sum(axis=1) / average["Total"]
    if ((frame["p"] < 0) | (frame["p"] > 1 + 1e-6)).any():
        raise ValueError("Stage fractions are inconsistent with the total timing")
    frame["limit"] = 1 / (1 - frame["p"].clip(upper=1 - 1e-12))
    frame.to_csv(RESULTS / "amdahl.csv", index=False)
    shown = frame.copy()
    shown.insert(0, "Cấu hình", shown["site"] + ", " + shown["resolution"])
    shown = shown[["Cấu hình", "p", "limit"]].rename(columns={"p": "p = (S2+S4+S6)/Total", "limit": "Giới hạn 1/(1−p)"})
    show_table(shown, "Ước lượng Amdahl khi tăng tốc " + ", ".join(columns), left=("Cấu hình",),
               formats={"p = (S2+S4+S6)/Total": "{:.3f}", "Giới hạn 1/(1−p)": "{:.2f}×"})
    by_size = frame.groupby("resolution", sort=False).agg(p_min=("p", "min"), p_max=("p", "max"),
                                                         limit_min=("limit", "min"), limit_max=("limit", "max")).loc[list(SIZES)]
    show_table(by_size.reset_index().rename(columns={"resolution": "Kích thước", "p_min": "p nhỏ nhất", "p_max": "p lớn nhất",
                                                     "limit_min": "Giới hạn nhỏ nhất", "limit_max": "Giới hạn lớn nhất"}),
               "Khoảng p và giới hạn speedup theo kích thước", left=("Kích thước",),
               formats={"p nhỏ nhất": "{:.3f}", "p lớn nhất": "{:.3f}", "Giới hạn nhỏ nhất": "{:.2f}×", "Giới hạn lớn nhất": "{:.2f}×"})
    return frame


def plot_amdahl(frame):
    figure, axis = plt.subplots(figsize=(9, 3.8))
    width = 0.26
    site_colors = {"site1": "#4C72B0", "site2": "#55A868", "site3": "#8172B2"}
    base = np.arange(len(SIZES))
    for index, site in enumerate(BENCHMARK_SITES):
        values = frame[frame.site == site].set_index("resolution").loc[list(SIZES), "limit"].values
        bars = axis.bar(base + (index - 1) * width, values, width, label=site, color=site_colors[site])
        axis.bar_label(bars, [f"{v:.1f}×" for v in values], fontsize=8.5, padding=2)
    axis.set_xticks(base, [SIZE_LABELS[s] for s in SIZES])
    axis.set_ylabel("Speedup tối đa 1/(1−p)")
    axis.set_title("Giới hạn Amdahl khi tăng tốc S2, S4, S6", loc="left")
    axis.legend(ncol=3, loc="upper right")
    axis.set_ylim(0, frame["limit"].max() * 1.18)
    figure.tight_layout()
    figure.savefig(RESULTS / "amdahl.png", bbox_inches="tight")
    plt.show()


# ---------------------------------------------------------------- 6. dataset
def show_dataset():
    manifest = pd.read_csv(DATASET / "manifest.csv")
    rows = []
    for pair in PAIRS:
        part = manifest[manifest.site == pair]
        left = part[part.side == "left"].iloc[0]
        right = part[part.side == "right"].iloc[0]
        rows.append({"Site": pair, "Cặp": left["pair"], "Ảnh gốc trái / phải": f"{left['source']} / {Path(right['source']).name}",
                     "Khung cắt 16:9 (x0 y0 x1 y1)": left["crop_box"], "Số ảnh": len(part),
                     "Kích thước": ", ".join(f"{w}×{h}" for w, h in sorted({(int(r.width), int(r.height)) for r in part.itertuples()}))})
    frame = pd.DataFrame(rows)
    show_table(frame, f"Dataset đã chuẩn bị: {len(manifest)} ảnh trong dataset/manifest.csv",
               left=("Site", "Cặp", "Ảnh gốc trái / phải", "Khung cắt 16:9 (x0 y0 x1 y1)", "Kích thước"))


# ---------------------------------------------------------------- 2. pipeline figure
def show_pipeline():
    figure, axis = plt.subplots(figsize=(12, 3.1))
    axis.set_xlim(0, 12.4)
    axis.set_ylim(0, 3.1)
    axis.axis("off")

    def box(x, y, text, color, w=1.55, h=0.62):
        axis.add_patch(FancyBboxPatch((x - w / 2, y - h / 2), w, h, boxstyle="round,pad=0.03,rounding_size=0.12",
                                      facecolor=color, edgecolor="none", alpha=0.9))
        axis.text(x, y, text, ha="center", va="center", fontsize=9,
                  color="white" if color in (STAGE_COLORS["S1"], STAGE_COLORS["S4"], STAGE_COLORS["S5"]) else "black")

    def arrow(start, end):
        axis.add_patch(FancyArrowPatch(start, end, arrowstyle="-|>", mutation_scale=11, color="#555", linewidth=1.2))

    for y, name in ((2.45, "A"), (0.65, "B")):
        box(0.6, y, f"Ảnh {name}\n(RGB)", "#DDDDDD", w=0.95)
        box(2.15, y, "S1\nảnh xám, blur", STAGE_COLORS["S1"])
        box(3.95, y, "S2\nkeypoint FAST", STAGE_COLORS["S2"])
        box(5.75, y, "S3\ndescriptor BRIEF", STAGE_COLORS["S3"])
        arrow((1.08, y), (1.37, y))
        arrow((2.93, y), (3.17, y))
        arrow((4.73, y), (4.97, y))
    box(7.6, 1.55, "S4\nghép cặp Hamming", STAGE_COLORS["S4"])
    box(9.35, 1.55, "S5\nH bằng RANSAC", STAGE_COLORS["S5"])
    box(11.1, 1.55, "S6\nwarp B, trộn", STAGE_COLORS["S6"])
    arrow((6.53, 2.3), (6.95, 1.75))
    arrow((6.53, 0.8), (6.95, 1.35))
    arrow((8.38, 1.55), (8.57, 1.55))
    arrow((10.13, 1.55), (10.32, 1.55))
    axis.text(11.1, 0.95, "→ panorama", ha="center", fontsize=9)
    axis.annotate("", xy=(11.1, 1.86), xytext=(0.6, 2.77),
                  arrowprops=dict(arrowstyle="-|>", color="#999", linestyle="--", linewidth=1,
                                  connectionstyle="arc3,rad=-0.12"))
    axis.text(6.6, 3.02, "ảnh màu A, B dùng lại ở S6", ha="center", fontsize=8.5, color="#777")
    axis.text(2.0, 1.55, "S1–S3 chạy riêng cho từng ảnh", fontsize=8.5, color="#777", ha="left")
    axis.text(7.6, 0.95, "S4–S6 cần cả hai ảnh", fontsize=8.5, color="#777", ha="center")
    figure.tight_layout()
    plt.show()

In [ ]:
RUNS = 5                        # timed runs per benchmark configuration (section 5.3)
ACCELERATED_STAGES = [2, 4, 6]  # stages planned for the GPU (section 7), used for Amdahl in 5.5

environment = check_runtime()
prepare_workspace()
show_session(environment, verify_inputs())

<a name="muc-1"></a>
## 1\. Thông tin nhóm và kế hoạch làm việc

### 1.1 Thành viên

| Họ tên | MSSV | Stage phụ trách |
|---|---|---|
| Nguyễn Hồ Anh Tuấn (nhóm trưởng) | 23120185 | S1 Tiền xử lý |
| Trần Hữu Kim Thành | 23120166 | S2 Phát hiện keypoint, S3 Descriptor |
| Lê Thượng Đế | 23120232 | S4 Ghép cặp, S5 Homography |
| Lê Hà Thanh Chương | 23120195 | S6 Warp và blend |

Mỗi stage có một người phụ trách. Người đó chọn phương pháp và tham số, viết tài liệu thiết kế, cài đặt và kiểm thử stage của mình. Các stage chỉ trao đổi dữ liệu qua kiểu dữ liệu chung trong `src/common.h`, nhờ vậy mỗi stage được kiểm thử riêng bằng dữ liệu tự tạo trước khi ghép thành pipeline.

### 1.2 Phân công theo phiên bản

| Thành viên | V0 tuần tự | V1 CUDA cơ bản | V2, V3 tối ưu |
|---|---|---|---|
| Tuấn | S1 | Kernel S1 | Tối ưu S1 nếu profiling V1 cho thấy S1 đáng kể |
| Thành | S2, S3 | Kernel S2, S3 | Tối ưu S2 (ưu tiên), S3 theo profiling |
| Đế | S4, S5 | Kernel S4, S5 | Tối ưu S4 (ưu tiên), S5 theo profiling |
| Chương | S6 | Kernel S6 | Tối ưu S6 (ưu tiên) |

S2, S4 và S6 chiếm phần lớn thời gian của V0 (mục 5.4), nên được ưu tiên tối ưu ở V2 và V3. Danh sách tối ưu cụ thể chỉ chốt sau khi có profiling của V1. Phương pháp và tham số của mỗi stage giữ nguyên từ V0 đến V3 để speedup giữa các phiên bản so sánh được.

### 1.3 Kế hoạch đến giữa kỳ

Nhóm chia phần việc giữa kỳ thành năm giai đoạn: chuẩn bị (chụp ảnh, thống nhất interface), cài đặt từng stage, ghép và kiểm chứng V0, đo và phân tích, hoàn thiện báo cáo. Công việc được theo dõi trên repo GitHub riêng của nhóm. Code của mỗi stage được review qua pull request trước khi vào nhánh chính. Mọi lần build, test và đo chính thức đều chạy trên Google Colab với GPU Tesla T4.

<a name="muc-2"></a>
## 2\. Bài toán ghép ảnh panorama

### 2.1 Bài toán

Đầu vào là hai ảnh chụp cùng một cảnh, có một phần nội dung chung. Gọi ảnh trái là A, ảnh phải là B. Đầu ra là một ảnh panorama chứa cả hai ảnh, trong đó vùng chung của A và B chồng khít lên nhau.

Hai ảnh được chụp từ cùng một chỗ đứng, người chụp chỉ xoay máy. Khi tâm máy ảnh không dời đi, mỗi điểm của ảnh B liên hệ với điểm tương ứng trong ảnh A qua một phép biến đổi phối cảnh (homography) H, là một ma trận 3×3. Bài toán ghép ảnh vì vậy quy về hai việc: tìm H, rồi dùng H đưa B vào hệ tọa độ của A. Nếu người chụp bước sang ngang, vật ở gần và vật ở xa dịch đi những khoảng khác nhau (parallax). Khi đó không có H nào khớp cho toàn ảnh và panorama sẽ bị lệch hoặc có ảnh kép. Đây là lý do đề yêu cầu "xoay máy, không bước đi" và chồng lấn khoảng 30–50%.

H có 8 bậc tự do, nên chỉ cần 4 cặp điểm tương ứng, không có 3 điểm nào thẳng hàng, là giải được. Cái khó là chương trình phải tự tìm các cặp điểm đó trong hàng triệu pixel, và một phần các cặp tìm được sẽ sai.

### 2.2 Pipeline sáu stage

In [ ]:
show_pipeline()

1. **S1 Tiền xử lý.** Ảnh màu được đổi sang ảnh xám vì các bước sau chỉ cần độ sáng, sau đó được làm mượt bằng bộ lọc Gaussian. Nếu không làm mượt, nhiễu ở từng pixel dễ bị nhận nhầm là góc ở S2.
2. **S2 Phát hiện keypoint.** Tìm những điểm dễ nhận ra lại trong ảnh kia, thường là góc, nơi độ sáng thay đổi theo nhiều hướng. Điểm trên vùng trơn hoặc dọc một cạnh thẳng thì không dùng được, vì nhiều vị trí lân cận trông giống hệt nhau.
3. **S3 Descriptor.** Gắn cho mỗi keypoint một dãy bit tóm tắt vùng ảnh nhỏ quanh nó. Hai keypoint nhìn cùng một chi tiết của cảnh sẽ có descriptor gần nhau.
4. **S4 Ghép cặp.** Với mỗi descriptor của A, tìm descriptor gần nhất trong B. Cặp nào mơ hồ, tức descriptor gần nhất và gần nhì trong B gần như bằng nhau, thì bị loại, vì vùng lặp lại như ô gạch hay cửa sổ dễ sinh cặp sai.
5. **S5 Ước lượng homography.** Danh sách cặp sau S4 vẫn còn cặp sai, nên không giải H trực tiếp từ mọi cặp được. RANSAC lặp nhiều lần: chọn ngẫu nhiên 4 cặp, giải ra một H, rồi xem bao nhiêu cặp khác khớp với H đó (inlier). H tốt nhất được tính lại từ toàn bộ inlier của nó.
6. **S6 Warp và blend.** Từ H và kích thước hai ảnh, tính khung panorama đủ chứa A và B sau biến đổi. Với mỗi pixel của khung, tính ngược ra vị trí tương ứng trong A và trong B, lấy màu bằng nội suy rồi trộn hai màu ở vùng chồng lấn.

S1 đến S3 chạy riêng trên từng ảnh, không phụ thuộc ảnh kia. S4 đến S6 cần dữ liệu của cả hai ảnh.

### 2.3 Dữ liệu giữa các stage

| Ranh giới | Dữ liệu |
|---|---|
| Vào S1 | Ảnh RGB 8 bit, đọc bằng stb_image |
| S1 → S2, S3 | Ảnh xám `float` trong khoảng [0, 1] |
| S2 → S3 | Danh sách keypoint: tọa độ pixel và điểm số |
| S3 → S4 | Một descriptor nhị phân 256 bit cho mỗi keypoint |
| S4 → S5 | Danh sách cặp chỉ số (keypoint A, keypoint B) kèm khoảng cách Hamming |
| S5 → S6 | Ma trận H 3×3 đưa điểm của B sang hệ tọa độ của A |
| Ra S6 | Ảnh panorama RGB và vị trí gốc tọa độ của A trong panorama |

### 2.4 Cách đánh giá

Một panorama tốt khi vùng chồng lấn khớp, không có cạnh bị đứt hay ảnh kép. Nhìn bằng mắt thì không đo được sai số, nên nhóm dùng thêm cặp ảnh synthetic: lấy một ảnh thật làm cảnh, tự tạo ảnh B từ ảnh A bằng một H biết trước, chạy pipeline rồi so H tìm được với H thật (mục 5.1). Sai số tính bằng pixel: khoảng cách giữa hai vị trí mà H thật và H ước lượng chiếu cùng một điểm của B vào A.

Về thời gian, đề yêu cầu đo riêng từng stage ở ba kích thước 1280×720, 1920×1080 và 3840×2160. Từ bảng đo, nhóm xác định stage tốn thời gian nhất và ước lượng speedup tối đa theo định luật Amdahl trước khi chuyển sang CUDA (mục 5.3 đến 5.5).

<a name="muc-3"></a>
## 3\. Phương pháp cho từng stage

| Stage | Phương pháp |
|---|---|
| S1 | Ảnh xám BT.601, Gaussian blur tách hai chiều |
| S2 | FAST-9, score V, non-maximum suppression 3×3, giữ N điểm mạnh nhất |
| S3 | BRIEF 256 bit không xoay |
| S4 | Brute-force 2 láng giềng gần nhất theo Hamming, ratio test |
| S5 | RANSAC 4 điểm, DLT chuẩn hóa, điểm MSAC, dừng thích nghi, refit trên inlier |
| S6 | Warp ngược bằng H⁻¹, nội suy bilinear, trộn 50/50 |

Mỗi mục dưới đây nêu phương pháp, tham số chính, lý do chọn theo chất lượng và theo khả năng song song trên GPU. Giá trị của các tham số được đọc lại từ mã nguồn và in ra ở mục 4.6. Một bộ tham số dùng cho cả ba độ phân giải; riêng ngưỡng inlier của RANSAC tỷ lệ theo kích thước ảnh.

### 3.1 S1 Tiền xử lý

#### Phương pháp và tham số

S1 đổi ảnh RGB sang ảnh xám theo hệ số BT.601, $Y = (0.299R + 0.587G + 0.114B)/255$, lưu kiểu `float` trong $[0, 1]$. Sau đó ảnh được làm mượt bằng Gaussian blur tách hai chiều: một lượt ngang rồi một lượt dọc, σ = 1 pixel, bán kính 3 (kernel 7 tap). Trọng số kernel tính bằng `double`, chuẩn hóa để tổng bằng 1 rồi ép về `float`. Pixel ngoài biên lấy bằng pixel biên gần nhất (clamp). Không dùng image pyramid.

```text
for each pixel (x, y):  gray(x, y) = (0.299 R + 0.587 G + 0.114 B) / 255
for each pixel (x, y):  temp(x, y) = sum_{k=-r..r} w[k] * gray(clamp(x + k), y)
for each pixel (x, y):  out(x, y)  = sum_{k=-r..r} w[k] * temp(x, clamp(y + k))
```

Với ảnh $W \times H$ và bán kính $r$, mỗi lượt blur tốn $O(WH(2r+1))$ phép nhân cộng và cần thêm một buffer `float` cỡ $WH$. Mỗi pixel cộng các tap theo thứ tự cố định từ $k=-r$ đến $k=r$, nên chạy lại cho kết quả giống từng bit.

#### Lý do chọn

FAST ở S2 và BRIEF ở S3 chỉ dùng độ sáng, nên ảnh xám là đủ. BT.601 giữ được cạnh giữa hai vùng khác màu nhưng có cùng tổng $R+G+B$, còn trung bình cộng ba kênh thì làm mất cạnh đó. Chi phí hai cách gần như bằng nhau vì S1 bị giới hạn bởi băng thông bộ nhớ.

Blur bớt nhiễu ở từng pixel, loại nhiễu mà FAST dễ nhận nhầm thành góc. Gaussian 2D kích thước $k \times k$ tốn $k^2$ phép nhân cộng mỗi pixel; bản tách hai chiều cho cùng kết quả (chỉ khác ở làm tròn số thực) với $2k$ phép. σ = 1 được chọn nhỏ vì FAST xét vòng tròn bán kính 3 pixel, blur mạnh hơn sẽ làm mất góc. Không dùng pyramid vì hai ảnh của cặp cùng độ phân giải và gần cùng tỷ lệ. Clamp không làm tối mép ảnh như đệm số 0, nên không sinh keypoint giả ở mép.

#### Khả năng song song

Cả ba lượt đều tính độc lập trên từng pixel. V1 dùng một thread cho một pixel output ở mỗi lượt, trọng số kernel đặt trong constant memory, ảnh xám giữ trên GPU cho S2. Hai lượt blur là stencil một chiều, ở V2/V3 có thể dùng shared memory tile có halo hoặc gộp với bước đổi ảnh xám.

### 3.2 S2 Phát hiện keypoint

#### Phương pháp và tham số

S2 dùng FAST-9 trên ảnh đã làm mượt của S1. Một pixel $p$ là góc khi có ít nhất 9 pixel liên tiếp trên vòng tròn 16 điểm bán kính 3 cùng sáng hơn, hoặc cùng tối hơn, tâm một ngưỡng $t = 0.05$ (tức 12.75/255). Mỗi góc được chấm bằng score V của Rosten và Drummond:

$$V = \max\Big(\sum_{\text{sáng}} (d_i - t),\ \sum_{\text{tối}} (-d_i - t)\Big), \qquad d_i = I_i - I_p.$$

Sau đó non-maximum suppression (NMS) 3×3 giữ các điểm có score lớn nhất trong lân cận; hai điểm bằng score thì điểm đứng trước theo thứ tự quét ảnh thắng. Cuối cùng giữ tối đa N = 2000 điểm mỗi ảnh, xếp theo score giảm dần, bằng nhau thì theo chỉ số quét $y \cdot W + x$ tăng dần. Bỏ lề 15 pixel ở mỗi cạnh ảnh để vòng FAST và patch 31×31 của S3 không ra ngoài ảnh.

```text
for each pixel p in [15, W-16] x [15, H-16]:
    d_i = I(p + offset_i) - I(p), i = 0..15
    if 9 consecutive d_i > t or 9 consecutive d_i < -t:  score(p) = V
for each p with score(p) > 0:
    keep p if no 8-neighbour has a larger score (ties: earlier raster index wins)
sort kept points by (score desc, raster index asc); output the first N
```

Segment test và NMS tốn $O(WH)$ với hằng số 16; bước sắp xếp tốn $O(M \log M)$ với $M$ là số điểm còn lại sau NMS. Khóa sắp xếp (score, chỉ số quét) không trùng nhau nên thứ tự output là duy nhất.

#### Lý do chọn

Hai ảnh của cặp được chụp đứng yên và xoay người, nên cùng độ phân giải và gần cùng tỷ lệ; chưa cần bất biến tỷ lệ. Nhóm đã xét FAST-12, Harris, Shi-Tomasi, FAST xếp hạng bằng Harris như ORB và FAST nhiều tỷ lệ. FAST-9 rẻ nhất, và Rosten cùng cộng sự (2010) cho thấy $n = 9$ lặp lại tốt hơn các giá trị $n$ khác. Score V có sẵn từ segment test nên không cần thêm bước Harris.

Ngưỡng 0.05 nằm giữa hai giá trị mặc định của OpenCV (10/255 cho `FastFeatureDetector`, 20/255 cho ORB), vì ảnh đã qua blur σ = 1 nên tương phản ở góc thấp hơn ảnh gốc. N = 2000 giữ S4 ở mức 2000 × 2000 phép so sánh mỗi cặp ảnh mà vùng chồng lấn vẫn đủ keypoint. Hệ quả của việc dùng cùng N cho mọi kích thước là ở 3840×2160, 2000 keypoint chỉ là một phần nhỏ của các góc tìm được và tập trung ở các góc tương phản mạnh nhất.

#### Khả năng song song

Segment test và NMS độc lập trên từng pixel, nên V1 dùng một thread cho một pixel ở mỗi bước. Ứng viên sau NMS được ghi bằng bộ đếm atomic; thứ tự ghi không ảnh hưởng kết quả vì bước sau sắp xếp theo khóa duy nhất, nên output giống V0. Bước chọn top-N cần sắp xếp trên GPU.

### 3.3 S3 Descriptor

#### Phương pháp và tham số

S3 tính descriptor BRIEF 256 bit, không xoay, cho mỗi keypoint của S2. Bit thứ $k$ bằng 1 khi điểm thứ nhất của cặp $k$ tối hơn điểm thứ hai, trên một bản sao ảnh xám được làm mượt thêm bằng Gaussian σ = 2, bán kính 4 (9 tap, dùng chung code blur với S1). 256 cặp điểm cố định nằm trong patch 31×31 quanh keypoint, lấy theo kiểu G II của bài báo BRIEF: hai điểm của mỗi cặp độc lập theo phân phối Gaussian đẳng hướng σ = 31/5 = 6.2, làm tròn và kẹp trong patch. Pattern được sinh một lần bằng SplitMix64 với seed 16 (script `tests/tool_gen_brief_pattern.py`) và lưu thành bảng hằng `src/brief_pattern.cpp`. 256 bit được đóng gói thành 8 word `uint32_t`, bit $k$ nằm ở word $k/32$, vị trí $k \bmod 32$. Không tính orientation.

```text
smoothed = Gaussian(sigma = 2, r = 4) of a copy of the S1 image
for each keypoint i in input order:
    (cx, cy) = (floor(x_i + 0.5), floor(y_i + 0.5))
    if the 31x31 patch leaves the image: drop keypoint i
    for k = 0..255:
        (ax, ay, bx, by) = pattern[k]
        bit k = smoothed(cx + ax, cy + ay) < smoothed(cx + bx, cy + by)
```

Làm mượt tốn $O(18\,WH)$ phép nhân cộng, phần descriptor tốn $O(256N)$ phép so sánh với $N$ keypoint. Keypoint bị bỏ được loại cùng lúc khỏi cả hai mảng, nên descriptor thứ $i$ luôn ứng với keypoint thứ $i$. Với lề 15 pixel của S2, không keypoint nào bị bỏ.

#### Lý do chọn

Máy được giữ ngang và chỉ xoay quanh trục dọc, nên hai ảnh gần như không xoay trong mặt phẳng ảnh. Vì vậy S3 không tính orientation như rotated BRIEF của ORB; bước đó chỉ tốn thêm chi phí mà không giúp gì cho dataset này. Descriptor nhị phân 256 bit cho phép S4 ghép bằng khoảng cách Hamming. Descriptor số thực kiểu SIFT bền hơn với thay đổi tỷ lệ và ánh sáng, nhưng phải đổi S4 sang khoảng cách L2 và đổi interface chung.

BRIEF so sánh từng cặp pixel nên rất nhạy với nhiễu. Calonder và cộng sự (2010) làm mượt patch bằng Gaussian σ = 2 trên cửa sổ 9×9 trước khi so sánh. S1 chỉ blur σ = 1 để giữ góc cho FAST, nên S3 tự làm mượt thêm trên bản sao riêng. Bài báo BRIEF cho thấy cách lấy mẫu G II tốt hơn các cách lấy ngẫu nhiên khác trong phần lớn thí nghiệm. Nhóm tự sinh pattern thay vì dùng bảng học sẵn của OpenCV để không phụ thuộc dữ liệu bên ngoài và giải thích được từng bước sinh.

#### Khả năng song song

Bước làm mượt dùng lại hai kernel blur của S1. Các descriptor độc lập với nhau, nên V1 dùng một thread cho một keypoint; bảng pattern 1 KB đặt trong constant memory. Bước loại keypoint ở biên dùng prefix sum để giữ đúng thứ tự. S3 chỉ có phép so sánh, nên nếu blur trên GPU giống V0 từng bit thì descriptor cũng giống từng bit.

### 3.4 S4 Ghép cặp

#### Phương pháp và tham số

S4 tìm chính xác hai láng giềng gần nhất bằng cách duyệt hết (brute-force), theo khoảng cách Hamming giữa hai descriptor 256 bit, theo chiều từ A sang B:

- Khoảng cách Hamming là số bit 1 sau khi XOR tám word của hai descriptor, từ 0 đến 256.
- Với mỗi descriptor A, chương trình duyệt mọi descriptor B và giữ hai láng giềng có chỉ số B khác nhau. Hai láng giềng bằng khoảng cách vẫn được giữ, để ratio test nhận ra trường hợp mơ hồ. Bằng khoảng cách thì chỉ số B nhỏ hơn đứng trước.
- Ratio test với ngưỡng 0.8: cặp được nhận khi $d_1 < 0.8\, d_2$, cài bằng phép so sánh số nguyên $5d_1 < 4d_2$. So sánh số nguyên tránh việc `0.8f` không biểu diễn chính xác làm sai các trường hợp đúng ranh giới. Khi $d_1 = d_2 = 0$, cặp bị loại.
- Không dùng cross-check. Nhiều điểm A được phép chọn cùng một điểm B; các cặp còn sai về hình học để S5 xử lý.
- Output xếp theo chỉ số A tăng dần. S4 báo lỗi nếu còn dưới 4 cặp, vì S5 cần ít nhất 4 cặp.

```text
for i in indices(A):
    best, second = none
    for j in indices(B):
        d = hamming(A[i], B[j])          # popcount of 8 XOR-ed words
        update (best, second) with (d, j), ordered by (distance, j)
    if 5 * best.distance < 4 * second.distance:
        append Match(i, best.index, best.distance)
```

Với $n_A$, $n_B$ descriptor, S4 xét $n_A n_B$ cặp, mỗi cặp 8 word, nên chi phí là $O(8\,n_A n_B)$ và không phụ thuộc trực tiếp vào số pixel. V0 đếm bit bằng vòng `bits &= bits - 1`, không dùng SIMD; mỗi A chỉ giữ hai láng giềng tốt nhất trong lúc duyệt, không lưu ma trận khoảng cách.

#### Lý do chọn

Hamming là khoảng cách tự nhiên của descriptor nhị phân. Duyệt hết bảo đảm tìm đúng hai láng giềng gần nhất theo khoảng cách, nên dễ kiểm chứng bằng dữ liệu có đáp án. Ratio test theo ý tưởng của Lowe (2004, mục 7.1) loại các cặp mà lựa chọn tốt nhất và tốt nhì gần như ngang nhau; bài báo dùng SIFT, nên ngưỡng của bài báo chỉ là điểm xuất phát, không phải bảo đảm chất lượng cho BRIEF. Các phương án đã xét: LSH và các cách tìm gần đúng nhanh hơn nhưng có thể bỏ lỡ láng giềng gần nhất thật; cross-check thêm điều kiện hai chiều và loại bớt cặp, nhưng S5 đã xử lý cặp sai. Vân lặp lại, ảnh mờ và vùng ít chi tiết vẫn có thể gây cặp sai hoặc thiếu cặp; ratio test không chứng minh một cặp là đúng về hình học.

#### Khả năng song song

Các cặp (A, B) độc lập với nhau; trên GPU mỗi word dùng XOR và `__popc`. V1 dự kiến một block cho một descriptor A, các thread chia nhau duyệt B, sau đó reduction trong block lấy hai láng giềng gần nhất. Kết quả được ghi theo thứ tự chỉ số A bằng prefix sum, không dùng atomic append, để giống V0. S4 là hot stage ở small và medium (mục 5.4), nên có lý do ưu tiên tối ưu.

### 3.5 S5 Homography

#### Phương pháp và tham số

S5 ước lượng H biến điểm của B sang hệ tọa độ của A bằng RANSAC 4 điểm. Mỗi lần thử, chương trình chọn ngẫu nhiên 4 cặp khác nhau, giải H bằng DLT có chuẩn hóa tọa độ, rồi chấm H đó trên toàn bộ danh sách cặp.

- Chọn mẫu: đều, không lặp trong một mẫu; bộ sinh SplitMix64 với seed 16, mỗi lần thử có luồng số ngẫu nhiên riêng theo (seed, số thứ tự lần thử). Mẫu có điểm trùng, có ba điểm gần thẳng hàng hoặc giải lỗi thì bị bỏ.
- Giải H: tọa độ A và B được chuẩn hóa riêng (trừ trọng tâm, co giãn để khoảng cách trung bình bình phương tới gốc bằng $\sqrt{2}$), lập hệ DLT, giải bằng SVD Jacobi một phía do nhóm tự cài, tính bằng `double`, rồi khử chuẩn hóa.
- Chấm điểm MSAC: loss của H là $\sum_j \min(e_j^2, \tau^2)$, với $e_j$ là sai số chiếu cặp $j$ từ B sang A. Cặp có $e_j^2 < \tau^2$ là inlier. Loss nhỏ hơn thì H tốt hơn; bằng nhau thì nhiều inlier hơn, rồi lần thử sớm hơn.
- Ngưỡng inlier τ: 3 pixel ở 1920×1080, tỷ lệ theo đường chéo ảnh nên là 2, 3, 6 pixel ở small, medium, large.
- Dừng thích nghi: tối đa 2000 lần thử; với tỷ lệ inlier $w$ của H tốt nhất hiện tại, số lần thử cần là
$$K = \left\lceil \frac{\ln(1-c)}{\ln(1-w^4)} \right\rceil, \qquad c = 0.99.$$
- Refit: tính lại H từ toàn bộ inlier của H tốt nhất, phân loại lại inlier, và tính lại một lần nữa nếu tập inlier thay đổi.
- Output: H dạng `float[9]` theo hàng, chia cho hệ số có trị tuyệt đối lớn nhất và cố định dấu.

Chi phí chấm điểm là $O(Km)$ với $K$ lần thử và $m$ cặp; giải H cho 4 điểm có kích thước cố định, refit tốn thêm chi phí theo số inlier.

#### Lý do chọn

RANSAC loại các cặp sai mà S4 còn để lọt. MSAC phân biệt được hai H có cùng số inlier nhờ mức sai số, thay vì chỉ đếm inlier. Chuẩn hóa tọa độ trước DLT (Harker và O'Leary, 2005) giúp hệ phương trình ổn định về số học khi tọa độ pixel lớn. Brown và Lowe (2007) dùng cùng hướng RANSAC với DLT cho bài toán ghép ảnh. Các phương án đã xét gồm PROSAC (cần xếp hạng chất lượng cặp để chọn mẫu), MAGSAC++ và VSAC (thêm cơ chế kiểm tra và tinh chỉnh phức tạp hơn). Nhóm chọn phương án đơn giản để tự cài, dễ tạo dữ liệu kiểm thử và giữ kết quả GPU giống CPU; nhóm chưa có thí nghiệm nào cho thấy phương án này tốt hơn các phương án kia. Ít inlier, điểm tập trung một vùng, parallax và vật chuyển động vẫn có thể làm H sai dù bước giải thành công; vì vậy độ chính xác được đo bằng cặp synthetic ở mục 5.1, không dựa vào số inlier.

#### Khả năng song song

Các lần thử và sai số của từng cặp tính độc lập được, nhưng việc chọn H tốt nhất và điều kiện dừng phải xét theo đúng thứ tự lần thử như V0. V1 dự kiến một block cho một lần thử: thread 0 sinh mẫu và giải DLT, các thread chia nhau tính sai số, reduction cộng loss theo cùng cây cộng cố định của V0. S5 chiếm phần nhỏ trong thời gian V0 (mục 5.3), nên chưa ưu tiên tối ưu sâu trước khi có profiling V1.

### 3.6 S6 Warp và blend

#### Phương pháp và tham số

S6 nhận hai ảnh màu gốc A, B và H của S5, rồi tạo panorama trên một canvas chung, gồm bốn bước:

1. Canvas và offset. Canvas là hình chữ nhật bao bốn góc của A và bốn góc của B sau khi chiếu qua H: $W_c = \lceil x_{\max} \rceil - \lfloor x_{\min} \rfloor + 1$, tương tự cho chiều cao; offset là $(-\lfloor x_{\min} \rfloor, -\lfloor y_{\min} \rfloor)$. Canvas bị từ chối nếu một chiều vượt 10 000 pixel hoặc diện tích vượt 100 triệu pixel.
2. Chép A vào canvas tại vị trí offset.
3. Warp ngược B: với mỗi pixel canvas, chiếu ngược về B bằng $H^{-1}$ và lấy màu bằng nội suy bilinear trên 4 pixel lân cận, tính bằng `double`.
4. Trộn: chỉ A phủ thì lấy màu A, chỉ B phủ thì lấy màu B, cả hai phủ thì lấy trung bình $(C_A + C_B)/2$, không ảnh nào phủ thì để đen. Giá trị được làm tròn một lần khi ghi ra 8 bit.

Một pixel nguồn hợp lệ khi tọa độ chiếu về nằm trong ảnh đó. Tính hợp lệ được xác định bằng hình học, không suy từ màu, nên pixel đen có thật trong ảnh vẫn được trộn bình thường.

Trước khi kiểm tra biên, tọa độ cách số nguyên gần nhất không quá `1e-6` pixel được đưa về số nguyên đó. Bilinear chặn các chỉ số láng giềng tại hàng/cột cuối để tránh đọc ngoài ảnh. Sau nội suy và blending, màu được clamp về `[0,255]` rồi làm tròn bằng `floor(value + 0.5)` khi ghi `uint8_t`. Đây là các quy ước của bản cài đặt nhóm.

```text
normalize H, invert it, check the condition estimate
project the 4 corners of B; compute canvas size and offset; reject a canvas over the limits
allocate a black canvas and copy A at the offset
for each canvas pixel (xc, yc):
    pB = H^-1 * (xc - ox, yc - oy)
    if pB lies inside B:
        cB = bilinear(B, pB)
        pixel = (A covers this pixel) ? (cA + cB) / 2 : cB
```

Với canvas $P$ pixel, chi phí là $O(P)$ và bộ nhớ thêm chỉ là ảnh output $3P$ byte; không lưu ảnh B đã warp riêng.

#### Lý do chọn

Warp ngược cho mỗi pixel đích đúng một giá trị, không có lỗ hay pixel bị ghi đè nhiều lần như warp xuôi. Bilinear mượt hơn nearest neighbour mà chỉ cần 4 mẫu. Nhóm tự tính bilinear để V0 và GPU dùng cùng công thức, vì bilinear phần cứng của texture chỉ có 8 bit cho phần lẻ của trọng số và không bảo đảm kết quả trùng bit với V0.

Các cách trộn đã xét gồm trung bình 50/50, feather theo khoảng cách tới biên ảnh, seam finding kết hợp multi-band, và adaptive blending. Thiết kế ban đầu chọn feather; từ ngày 13/10 nhóm chuyển sang trung bình 50/50 vì cách này tốn $O(1)$ mỗi pixel, không cần thêm buffer, và dễ giữ cùng một công thức từ V0 đến V3. Giới hạn của nó là không che được sai lệch: nếu H sai, cảnh có parallax hoặc vật chuyển động thì vùng chồng lấn sẽ có ảnh kép, và chênh sáng giữa hai ảnh tạo đường chuyển tiếp. Seam finding và multi-band giảm được các lỗi này nhưng cần thêm mask, nhiều tầng ảnh và bước tối ưu đường nối.

#### Khả năng song song

Trong lượt gather B và blend, mỗi pixel canvas được xử lý độc lập; toàn bộ V0 vẫn gồm các lượt khởi tạo canvas, chép A và cập nhật từ B, nên V1 dùng một thread cho một pixel canvas, không cần atomic. Phần tính canvas và nghịch đảo H rất nhỏ, chạy trong một kernel một thread; Host chỉ đọc lại kích thước canvas để cấp phát output. Rủi ro chính là S6 dùng `double`, mà T4 có throughput phép tính 64 bit thấp hơn nhiều so với 32 bit.



<a name="muc-4"></a>
## 4\. Phiên bản tuần tự V0

### 4.1 Chương trình và dòng lệnh

V0 là chương trình C++ đơn luồng chạy đủ sáu stage, build thành một file thực thi `stitch` từ thư mục `src/`. Cách gọi:

```text
stitch <left> <right> <output> --version 0 [--runs N] [--debug]
```

Chương trình đọc hai ảnh bằng `stb_image`, chạy một lần warm-up không đo, sau đó chạy `N` lần có đo (mặc định 5). Mỗi lần đo in một dòng CSV ra stdout gồm thời gian S1 đến S6 và Total (ms), cuối cùng là hai dòng `Average` và `Minimum`. Panorama của lần chạy cuối được ghi ra file bằng `stb_image_write`. Cờ `--debug` in cấu hình, số keypoint, số match, thông tin RANSAC, H và kích thước canvas ra stderr sau khi các lần đo đã xong, nên không ảnh hưởng thời gian đo. Mục 4.6 có ví dụ output của chương trình.

### 4.2 Cấu trúc mã nguồn

| Nhóm file trong `src/` | Nội dung |
|---|---|
| `common.h`, `versions.h`, `cuda_utils.cuh`, `stb_impl.cpp` | Kiểu dữ liệu chung (`Image`, `GrayImage`, `Keypoint`, `Descriptor`, `Match`, `Homography`, `Panorama`, `Config`, `TimingStats`, `ErrorCode`), chọn phiên bản, tiện ích CUDA cho các phiên bản sau, phần cài đặt của stb |
| `main.cu` | Đọc tham số dòng lệnh, đọc và ghi ảnh, warm-up, vòng đo, in CSV và báo lỗi |
| `v0_config.*`, `v0_pipeline.*` | Cấu hình V0 và hàm chạy lần lượt S1 đến S6 trên vùng nhớ dùng lại giữa các lần chạy |
| `v0_preprocess.*`, `gaussian_kernel.*`, `gaussian_blur.*`, `image_layout.*` | S1, cùng phần blur và kiểm tra layout ảnh dùng chung với S2, S3 |
| `v0_detect.*`, `fast_corner.*`, `feature_border.h` | S2 |
| `v0_describe.*`, `brief_pattern.*` | S3 |
| `v0_match.*`, `feature_support.*` | S4 và các hằng số dùng chung của S4, S5 |
| `v0_homography.*`, `v0_homography_internal.h` | S5 |
| `v0_warp_blend.*` | S6 |

Mỗi stage nằm trong file riêng và chỉ giao tiếp qua kiểu dữ liệu trong `common.h`. Nhờ vậy từng stage được kiểm thử độc lập trước khi ghép, và các phiên bản GPU sau này thay từng stage mà không đổi phần còn lại.

### 4.3 Cách đo thời gian

Mỗi stage và toàn bộ pipeline được đo bằng `std::chrono::high_resolution_clock`, theo đúng quy định của đề cho V0. Vùng đo không gồm đọc, ghi ảnh và log debug. Các buffer có kích thước biết trước (ảnh xám, mảng keypoint, descriptor, match) được cấp phát một lần trước vùng đo và dùng lại qua các lần chạy. Buffer tạm bên trong một stage và ảnh output của S6, có kích thước chỉ biết sau khi có H, được cấp phát trong stage nên tính vào thời gian stage đó. S1, S2, S3 chạy cho cả hai ảnh, thời gian ghi là tổng của hai ảnh. Total được đo riêng quanh cả pipeline chứ không lấy bằng tổng các stage; hai giá trị gần như bằng nhau.

### 4.4 Chạy tuần tự, tái lập và code bên ngoài

V0 không dùng OpenMP, `std::thread`, `std::async`, `std::execution` hay intrinsics SIMD, và không build với `-fopenmp`. Cờ `-fno-tree-vectorize` và `-fno-tree-slp-vectorize` tắt việc compiler tự vector hóa, để V0 là baseline đơn luồng không SIMD. Cờ `--fmad=false` và `-ffp-contract=off` tắt việc compiler tự động gộp phép nhân và cộng thành FMA; S6 vẫn sử dụng các lời gọi std::fma tường minh, để kết quả số thực của V0 làm chuẩn so sánh cho các phiên bản GPU.

Phần ngẫu nhiên duy nhất là chọn mẫu RANSAC ở S5, dùng seed cố định 16. Mọi phép cộng dồn có thứ tự cố định, nên chạy lại trên cùng input cho cùng output; mục 5.1 kiểm tra điều này bằng cách chạy lại một cặp ảnh và so từng pixel.

Code lấy từ bên ngoài chỉ có hai header `stb_image.h` (v2.30) và `stb_image_write.h` (v1.16) của Sean Barrett, lấy nguyên từ [github.com/nothings/stb](https://github.com/nothings/stb) để đọc và ghi ảnh; `stb_impl.cpp` chỉ bật phần cài đặt của hai header này. Toàn bộ code thuật toán do nhóm tự viết. Các file cài theo mô tả trong bài báo hoặc tài liệu (FAST, BRIEF, ratio test, DLT chuẩn hóa, SVD Jacobi, SplitMix64, MSAC) ghi nguồn ở dòng `Adapted from:` đầu file và trong mục 8.

### 4.5 Build từ `src/` của bài nộp

In [ ]:
build_program()

### 4.6 Tham số và output của chương trình

Bảng đầu đọc các hằng số trực tiếp từ các file trong `src/` vừa build, nên các tham số nêu ở mục 3 khớp với code thật. Sau đó cell chạy `stitch --runs 1 --debug` trên cặp site1 ở ba kích thước để in ví dụ output CSV và cấu hình mà chương trình dùng ở từng kích thước. Thời gian của các lượt chạy này chỉ để minh họa định dạng, không dùng cho benchmark ở mục 5.3.

In [ ]:
show_parameters()
demo_run()

### 4.7 Kiểm thử và xử lý lỗi

Mỗi stage có bộ test C++ riêng trong `tests/`, đáp án được tính độc lập với hàm đang kiểm tra (ví dụ FAST so với cài đặt vét cạn, blur tách hai chiều so với Gaussian 2D bằng `double`, DLT trên điểm có H biết trước). Hai test tích hợp chạy S1–S5 và S1–S6 trên một cặp ảnh dịch biết trước, test sau so cả pixel của panorama. Các test Python kiểm tra các script hỗ trợ: sinh cặp synthetic và tính sai số H, chuẩn bị dataset, chạy và tổng hợp benchmark. Cột "Số kiểm tra" là số assertion hoặc check trong mỗi bộ test, không phải số ảnh được thử.

In [ ]:
run_tests()

Năm test bị bỏ qua ở dòng cuối là các test kiểm tra cấu trúc của chính file notebook; trong thư mục chạy trên Colab không có file notebook nên các test này tự bỏ qua, còn trên máy của nhóm chúng được chạy đủ.

Mỗi stage trả về một `ErrorCode` thay vì gọi `exit()`. `main.cu` đổi mã lỗi thành thông báo và exit code khác 0, gồm các trường hợp: thiếu file, ảnh không đọc được, tham số dòng lệnh sai, phiên bản chưa có, `--runs` không dương, không đủ keypoint hoặc match, homography suy biến, canvas vượt giới hạn và cấp phát thất bại. Khi một stage lỗi, chương trình không ghi panorama. Cell dưới thử các lỗi dòng lệnh và lỗi file.

In [ ]:
run_cli_cases()

<a name="muc-5"></a>
## 5\. Kết quả và profiling V0

<a name="muc-5-1"></a>
### 5.1 Độ chính xác homography trên cặp synthetic

Đề yêu cầu đo sai số homography trên cặp synthetic so với ground truth. Phần này giải thích vì sao nhóm lấy ground truth bằng cách tự tạo cặp ảnh, cách tạo, cách đo, rồi đến kết quả.

#### Vì sao tự tạo cặp ảnh thay vì so với OpenCV hay ảnh panorama của điện thoại

Muốn biết H của V0 sai bao nhiêu thì phải biết H đúng. Với hai ảnh chụp thật, không ai biết H đúng, chỉ có các ước lượng của H. Nhóm đã xét ba cách lấy mốc so sánh.

**Cách 1: lấy H do OpenCV ước lượng làm mốc** (ví dụ ORB, `BFMatcher` và `findHomography` với RANSAC trên cùng cặp ảnh). OpenCV cũng là một bộ ước lượng như V0. Nó có detector, matcher, ngưỡng RANSAC và seed riêng, nên H của nó cũng có sai số, và sai số đó đổi theo tham số. Độ lệch giữa hai H vì vậy chỉ cho biết hai chương trình giống nhau đến đâu:

- Nếu V0 lệch OpenCV 2 pixel thì không biết bên nào sai, hay cả hai cùng sai.
- Hai chương trình có thể cùng sai theo một hướng. Trên cảnh có parallax, cả hai đều bị kéo về mặt phẳng có nhiều keypoint nhất, nên độ lệch giữa chúng nhỏ trong khi panorama vẫn có ảnh kép.
- Một cài đặt đúng nhưng chọn tham số khác OpenCV (ngưỡng ratio, số keypoint, tiêu chí dừng) vẫn cho H khác vài phần mười pixel, nên không có ngưỡng nào phân biệt được "khác OpenCV" với "sai".

Cách này hợp để đối chiếu nhanh khi gỡ lỗi, nhưng không phải ground truth. Nó cũng cần đưa thêm OpenCV vào quy trình kiểm chứng, trong khi chương trình chỉ dùng thư viện chuẩn và stb.

**Cách 2: so với ảnh chụp bằng chế độ panorama của điện thoại.** Điện thoại không ghép bằng một homography trên mặt phẳng ảnh A như V0. Ảnh được chiếu lên mặt trụ hoặc mặt cầu, ghép từ nhiều khung hình liên tiếp khác với hai ảnh của cặp, được cân bằng sáng, trộn bằng thuật toán riêng, cắt viền và thường đổi tỷ lệ. Hai ảnh panorama có thể khác nhau về hình học ngay cả khi cả hai đều "đúng". Vì vậy không suy ra được H từ ảnh của điện thoại, còn so từng pixel thì không có ý nghĩa. Cách này chỉ dùng để so cảm quan.

**Cách 3, nhóm chọn: tự tạo cặp ảnh từ một H biết trước.** Lấy một ảnh thật làm cảnh, cắt ra ảnh A, rồi tạo ảnh B bằng cách lấy mẫu cảnh đó theo một ma trận $H_{\mathrm{gt}}$ do nhóm đặt trước. Vì B được tạo từ $H_{\mathrm{gt}}$, nên $H_{\mathrm{gt}}$ là đáp án đúng theo định nghĩa, không phụ thuộc bộ ước lượng nào, kể cả code đang kiểm tra. Pipeline chạy đủ S1–S6 từ hai file ảnh như với ảnh thật, không nhận trước cặp điểm hay H nào. Đây cũng là cách đề yêu cầu: *"the homography error on the synthetic pair, compared against the ground truth"*.

Giới hạn của cách 3: ảnh B không có các khó khăn của ảnh chụp thật, gồm nhiễu cảm biến, chênh sáng giữa hai lần chụp, parallax, méo ống kính và vật chuyển động. Vì vậy kết quả synthetic cho thấy pipeline tính đúng hình học (đúng chiều H, đúng chuẩn hóa tọa độ, ước lượng được tới mức dưới một pixel), còn chất lượng trên ảnh thật được đánh giá riêng ở mục 5.2 bằng cách xem panorama và các vùng cắt ở độ phân giải gốc. Số inlier của S5 cũng không dùng làm ground truth, vì inlier được xác định bằng chính H ước lượng.

#### Cách tạo cặp ảnh

Ảnh nguồn là `raw/tuan/standard1_left.jpg` (8160×6144), SHA-256 được kiểm tra ở cell chuẩn bị. Script `scripts/make_synthetic_pair.py` thu nhỏ cảnh bằng Lanczos, lấy A là một vùng cắt, rồi tạo B bằng nội suy bilinear theo $H_{\mathrm{gt}}$ (chiều từ B sang A). Ảnh được lưu PNG để không thêm nhiễu nén JPEG. Hai loại hình học được tạo ở cả ba kích thước:

- **translation:** B dịch ngang $\operatorname{round}(0.55(W-1))$ pixel so với A, tức 703, 1055 và 2111 pixel ở small, medium, large.
- **projective:** trong tọa độ chia cho $W-1$, B xoay 3° quanh tâm, có thành phần phối cảnh (0.10, 0.05) và dịch (0.55, 0.02). Trường hợp này kiểm tra phần phối cảnh của H, thứ mà phép dịch không kiểm tra được.

#### Cách đo sai số và ngưỡng chấp nhận

Đặt một lưới 20 × 20 điểm trên B, giữ lại tập Q gồm các điểm mà $H_{\mathrm{gt}}$ chiếu vào trong A, tức các điểm thuộc vùng chồng lấn. Q chỉ phụ thuộc $H_{\mathrm{gt}}$, không phụ thuộc H ước lượng hay tập inlier. $H_{\mathrm{est}}$ là H dạng `float` mà S5 thật sự bàn giao cho S6, đọc từ log `--debug`. Với $\pi(H, q)$ là phép chiếu điểm $q$ bằng $H$:

$$e(q) = \big\lVert \pi(H_{\mathrm{est}}, q) - \pi(H_{\mathrm{gt}}, q) \big\rVert_2, \qquad
\mathrm{RMSE} = \sqrt{\tfrac{1}{|Q|}\textstyle\sum_{q \in Q} e(q)^2}, \qquad
\mathrm{Max} = \max_{q \in Q} e(q).$$

NRMSE là RMSE chia cho đường chéo ảnh A. Nếu có điểm nào của Q bị $H_{\mathrm{est}}$ chiếu ra vô cực, phép đánh giá thất bại, không bỏ điểm đó đi.

Nhóm đo sai số bằng pixel tại vùng chồng lấn chứ không so từng hệ số của H. H chỉ xác định tới một hệ số tỷ lệ, và các hệ số có đơn vị rất khác nhau: phần xoay và co giãn cỡ 1, phần dịch tính bằng pixel, phần phối cảnh cỡ $10^{-4}$. Hiệu hai hệ số vì vậy không cho biết panorama lệch bao nhiêu pixel. Sai số chiếu thì đo đúng thứ người xem thấy: một điểm của B bị đặt lệch bao xa trên panorama.

Ngưỡng chấp nhận do nhóm đặt trước khi chạy (đề không quy định số cụ thể): **mỗi cặp** phải có RMSE ≤ 0.5 px và Max ≤ 1 px.

- Max ≤ 1 px yêu cầu mọi điểm trong tập kiểm tra Q có sai số chiếu không quá một pixel. Q gồm 180 điểm ở trường hợp translation và 165 điểm ở trường hợp projective, được giữ lại từ lưới 20×20. Kết quả này kiểm chứng hình học trên tập điểm đã định trước, không phải trên mọi pixel trong vùng chồng lấn; chất lượng đường nối và màu sắc được đánh giá riêng bằng panorama và crop.
- RMSE ≤ 0.5 px là ngưỡng nghiệm thu nhóm chọn cho các fixture hiện tại, nhằm phát hiện sai lệch trong chiều H, chuẩn hóa tọa độ hoặc quy ước tâm pixel. Ngưỡng này không bảo đảm mọi bộ ước lượng được cài đúng đều đạt trên mọi dữ liệu, cũng không thay thế kiểm tra chất lượng trên ảnh thật.

Ngưỡng này khác ngưỡng inlier τ = 2, 3, 6 px của RANSAC. τ phải đủ rộng để nhận các cặp đúng trên ảnh thật có nhiễu, còn ngưỡng nghiệm thu đo H cuối cùng so với đáp án.

Cell dưới sinh sáu cặp, chạy pipeline trên từng cặp và tính sai số. Hình bên dưới bảng minh họa cặp projective ở medium cùng hai ma trận H.

#### Synthetic Fixture Metadata

Cell dưới in thông tin từ ảnh nguồn và các hằng số của script sinh dữ liệu, trước khi chạy kiểm chứng. Hai ảnh A và B được lấy từ cùng scene C; B được lấy mẫu qua H_gt. Kích thước, hash, phép dịch, góc xoay và hệ số projective được ghi cùng tập điểm Q để người đọc đối chiếu với mô tả ở trên. Ngưỡng RMSE/Max là ngưỡng nghiệm thu của nhóm, không phải kết quả đo.

In [ ]:
# Read the fixture definition; do not generate ground truth with the estimator under test.
import importlib.metadata as package_metadata
import make_synthetic_pair as synthetic_generator
import homography_error as synthetic_metric
from dataset_common import load_oriented
from midterm_notebook import source_manifest

fixture_source = PROJECT / RAW_SYNTHETIC_RELATIVE
fixture_digest = sha256(fixture_source)
if fixture_digest != EXPECTED_SYNTHETIC_SHA256:
    raise StepFailed("Synthetic source SHA-256 differs from the documented fixture.")
with PILImage.open(fixture_source) as image:
    stored_source_size = image.size
oriented_source, source_exif, source_format = load_oriented(fixture_source)
oriented_source_size = oriented_source.size
oriented_source.close()
FIXTURE_THRESHOLDS = {"rmse_px": 0.5, "max_px": 1.0}
SOURCE_SNAPSHOT = source_manifest(PROJECT)
BINARY_SNAPSHOT = sha256(STITCH)
SUPPORT_SNAPSHOT = sha256(SUPPORT_ARCHIVE) if SUPPORT_ARCHIVE.is_file() else None
show_facts([
    ("Ảnh nguồn", RAW_SYNTHETIC_RELATIVE),
    ("Kích thước lưu / sau EXIF", f"{stored_source_size} / {oriented_source_size}"),
    ("SHA-256 ảnh nguồn", fixture_digest),
    ("Hệ tọa độ", "H_gt: B → A; tâm pixel nguyên"),
    ("Lưới đánh giá", f"{synthetic_metric.GRID}×{synthetic_metric.GRID}; Q chọn theo H_gt"),
    ("Ngưỡng nghiệm thu", "RMSE ≤ 0.5 px; Max ≤ 1 px trên từng cặp"),
], "Nguồn và quy ước của cặp synthetic")
fixture_rows = []
for case in synthetic_generator.CASES:
    for size, (width, height) in SIZES.items():
        matrix = synthetic_generator.case_homography(case, width, height)
        xs, ys = synthetic_metric.check_points(matrix, width, height)
        projective_case = case == "projective"
        fixture_rows.append({
            "Trường hợp": case, "Kích thước": f"{size}: {width}×{height}",
            "Dịch x (đơn vị W−1)": synthetic_generator.OVERLAP_SHIFT,
            "Dịch x nguyên (px)": round(synthetic_generator.OVERLAP_SHIFT * (width - 1)) if not projective_case else "–",
            "Dịch y (đơn vị W−1)": synthetic_generator.VERTICAL_SHIFT if projective_case else 0,
            "Góc xoay (độ)": synthetic_generator.ROTATION_DEG if projective_case else 0,
            "Hệ số projective": str(synthetic_generator.PERSPECTIVE if projective_case else (0.0, 0.0)),
            "Điểm Q / lưới": f"{len(xs)}/{synthetic_metric.GRID ** 2}",
        })
show_table(pd.DataFrame(fixture_rows), "Thông số sinh sáu cặp; đọc từ script đang sử dụng",
           left=("Trường hợp", "Kích thước", "Hệ số projective", "Điểm Q / lưới"))
(RESULTS / "fixture_metadata.json").write_text(json.dumps({
    "source": RAW_SYNTHETIC_RELATIVE, "source_sha256": fixture_digest,
    "stored_size": stored_source_size, "oriented_size": oriented_source_size,
    "grid": synthetic_metric.GRID, "thresholds": FIXTURE_THRESHOLDS,
    "cases": fixture_rows,
}, ensure_ascii=False, indent=2), encoding="utf-8")

In [ ]:
synthetic_h = run_synthetic()
show_synthetic_example(synthetic_h)

#### Nhận xét về kết quả synthetic

Cả sáu cặp đều đạt ngưỡng. RMSE lớn nhất là 0.100222 px và Max lớn nhất là 0.209020 px, đều ở cặp projective large, tức mọi điểm kiểm tra lệch chưa tới một phần tư pixel. Ở trường hợp translation, B có cùng nội dung pixel với một phần của A nên sai số gần như bằng 0; trường hợp này kiểm tra chiều H và phép dịch, còn trường hợp projective mới kiểm tra phần xoay và phối cảnh. Sai số projective tăng theo kích thước ảnh vì cùng một sai số tương đối của H tạo sai số pixel lớn hơn trên ảnh lớn hơn; NRMSE (chia cho đường chéo) của ba kích thước đều cùng bậc $10^{-5}$.

Kết quả này cho thấy S1–S5 tìm được H đúng chiều và chính xác dưới một pixel trên dữ liệu có hình học kiểm soát. Nó không chứng minh H trên ảnh thật chính xác tới mức đó, vì ảnh thật có thêm nhiễu, chênh sáng và parallax; phần đó được xem ở mục 5.2.

Cell dưới chạy lại cặp projective small với cùng binary, input và cấu hình để kiểm tra tính tái lập của output (không so thời gian).

In [ ]:
check_repeatability()

<a name="muc-5-2"></a>
### 5.2 Panorama của các địa điểm

Cell dưới chạy V0 trên ba cặp chuẩn và cặp khó ở kích thước medium (1920×1080). Mỗi panorama được hiện ở dạng thu nhỏ để xem tổng thể, kèm ba vùng cắt 512×512 trong vùng chồng lấn hiện ở **độ phân giải gốc** (một pixel ảnh bằng một pixel màn hình), vì ảnh thu nhỏ che mất các lỗi nhỏ như cạnh nhòe hay ảnh kép. Ba vùng cắt nằm ở phần trên, giữa và dưới của hình chữ nhật bao vùng chồng lấn. Thời gian của các lượt chạy này không dùng cho benchmark.

Vùng đen quanh panorama là phần canvas không thuộc ảnh nào. Canvas là hình chữ nhật bao cả A và B sau khi warp, còn B sau khi warp là một tứ giác, nên luôn có vùng thừa. Đây là output đúng của S6 theo thiết kế. Nhóm không cắt bỏ vùng đen, vì cắt sẽ đổi phương pháp của S6 và làm mất một phần ảnh.

#### Các cặp ảnh đầu vào

Mỗi hàng là một cặp đầu vào cỡ medium: A ở bên trái, B ở bên phải. Ảnh được thu nhỏ chỉ để hiển thị; chương trình vẫn xử lý file gốc trong dataset. Gallery này giúp đối chiếu vùng chồng lấn và các vật gần máy trước khi xem panorama.

In [ ]:
# Display the prepared inputs without changing the images used by stitch.
for pair in PAIRS:
    figures = []
    for side, label in (("left", "A · ảnh trái"), ("right", "B · ảnh phải")):
        path = DATASET / pair / f"{PANORAMA_SIZE}_{side}.jpg"
        figures.append('<figure style="margin:0; flex:1; min-width:280px">'
                       + _image_tag(path, width=640)
                       + f'<figcaption style="font-size:13px">{html.escape(pair)} · {label}</figcaption></figure>')
    display(HTML('<div style="display:flex; flex-wrap:wrap; gap:12px; margin:8px 0 20px">'
                 + "".join(figures) + '</div>'))

#### Các cặp đặc trưng tương ứng và các điểm trong tập hợp khớp (inliers)

Minh họa site1 ở cỡ medium bằng công cụ `tests/tool_pipeline_trace.cpp`, link với đúng `src/` vừa build. Dấu đỏ trên ảnh xám là keypoint được giữ lại sau S3. Trong hình ghép cặp, đường xanh là inlier theo H của S5, đường đỏ là match bị S5 loại. Nhãn inlier thể hiện sự phù hợp với H ước lượng, không phải ground truth của ảnh thật.

Công cụ chạy ngoài benchmark, không thay ảnh, keypoint hoặc match của lần đo chính thức. Hình được thu nhỏ để trình bày; số lượng keypoint, descriptor và match lấy từ log của công cụ.

In [ ]:
# Reuse the untimed trace tool; no algorithm implementation is copied into the notebook.
TRACE_PAIR = "site1"
TRACE_SIZE = "medium"
trace_folder = RESULTS / "feature_trace"
trace_folder.mkdir(exist_ok=True)
trace_prefix = trace_folder / f"{TRACE_PAIR}_{TRACE_SIZE}"
trace_binary = BUILD / "tool_pipeline_trace"
trace_sources = sorted(str(path.relative_to(PROJECT)) for path in (PROJECT / "src").glob("*.cpp"))
run(["nvcc", *NVCC_FLAGS, "-Isrc", *trace_sources, "tests/tool_pipeline_trace.cpp", "-o", trace_binary],
    "feature_trace_build.log")
trace_result = run([trace_binary, DATASET / TRACE_PAIR / f"{TRACE_SIZE}_left.jpg",
                    DATASET / TRACE_PAIR / f"{TRACE_SIZE}_right.jpg", trace_prefix], "feature_trace.log")
display(HTML('<pre style="font-size:13px; white-space:pre-wrap">'
             + html.escape(trace_result.stdout) + '</pre>'))
keypoint_figures = []
for side, label in (("left", "A"), ("right", "B")):
    path = Path(str(trace_prefix) + f"_s2_{side}.png")
    keypoint_figures.append('<figure style="margin:0; flex:1; min-width:280px">'
                           + _image_tag(path, width=640)
                           + f'<figcaption style="font-size:13px">{TRACE_PAIR} · keypoint ảnh {label}</figcaption></figure>')
display(HTML('<div style="display:flex; flex-wrap:wrap; gap:12px">' + "".join(keypoint_figures) + '</div>'))
display(HTML('<figure style="margin:12px 0">'
             + _image_tag(Path(str(trace_prefix) + "_s5_matches.png"), width=1400)
             + '<figcaption style="font-size:13px">Xanh: S5 giữ; đỏ: S5 loại. Màu không thể hiện nhãn đúng/sai đã biết trước.</figcaption></figure>'))

In [ ]:
panorama_info = run_panoramas()

#### site1: nhà thờ, ngoài trời

In [ ]:
show_panorama("site1", "site1, nhà thờ")
show_crops("site1")

Panorama liền khung và phần kiến trúc ở xa khớp nhau trên ảnh tổng thể. Các vùng cắt ở độ phân giải gốc cho thấy tượng, dây điện và một số cạnh hàng rào, ghế có viền nhòe hoặc kép. Ảnh thu nhỏ che bớt các sai lệch này, nên chất lượng phải đánh giá trên vùng cắt. Chỉ từ các ảnh này chưa tách được phần nhòe do rung máy lúc chụp với phần lệch do warp và blend.

#### site2: Ẩn Coffee, trong nhà

In [ ]:
show_panorama("site2", "site2, Ẩn Coffee")
show_crops("site2")

Đây là cặp có lỗi rõ nhất. Trong vùng chồng lấn, khung cửa sổ và bức tường phía xa khớp nhau, nhưng các vật ở gần máy bị nhân đôi: đế đèn trên trần, các đèn thả, hàng ly trên quầy và cạnh quầy đều hiện hai bản lệch nhau. Đây là lỗi parallax. Cảnh trong nhà có nhiều mặt phẳng ở các khoảng cách khác nhau, trong khi một homography chỉ khớp được một mặt phẳng (ở đây là tường và cửa sổ). Khi tâm máy dịch đi một chút lúc xoay người chụp, vật ở gần dịch nhiều hơn vật ở xa. Trộn 50/50 giữ cả hai bản nên ảnh kép hiện rõ. Ngoài ra có một đường chuyển sáng ở mép trên của ảnh A trên trần nhà, do hai ảnh có độ sáng khác nhau.

Ảnh B ở site2 còn bị kéo giãn mạnh về phía phải. Khi góc xoay giữa hai lần chụp lớn, chiếu B lên mặt phẳng của A làm phần xa tâm A phóng to dần. Đây là giới hạn của panorama phẳng dùng một homography, không phải lỗi của H; panorama trụ hoặc cầu tránh được hiện tượng này nhưng không nằm trong pipeline của đề.

#### site3: tiệm cà phê Mai An, trong nhà

In [ ]:
show_panorama("site3", "site3, tiệm cà phê Mai An")
show_crops("site3")

Tường, trần, đèn chùm và khung tranh khớp tốt. Ghế và bàn ở tiền cảnh có viền mờ nhẹ, cùng lỗi parallax như site2 nhưng nhẹ hơn. Ở mép phải của ảnh A trên tường có một đường chuyển sáng thẳng đứng, do hai ảnh có độ sáng khác nhau.

#### Cặp khó: khu ký túc xá, ban đêm

In [ ]:
show_panorama("challenge", "Cặp khó, khu ký túc xá ban đêm")
show_crops("challenge")

Cặp này chụp ban đêm: chênh sáng lớn giữa bầu trời, tòa nhà có đèn và tán cây, có người đi bộ và xe máy di chuyển. Pipeline vẫn tìm được nhiều inlier nhất và tỷ lệ inlier cao nhất trong bốn cặp (391 trên 614 match, tỷ lệ 64%) và panorama khớp về hình học: tòa nhà, cột đèn, thùng rác và ghế đá đều liền mạch. Người đi bộ và xe máy nằm ngoài vùng chồng lấn nên không bị nhân đôi; xe máy bị nhòe do chuyển động lúc chụp. Các hạn chế nhìn thấy được: nhiễu ảnh đêm trên tán cây tối, một đường chuyển sáng trên bầu trời ở mép trên của ảnh A, và chân cột đèn có viền mờ nhẹ.

#### Đánh giá chung

V0 tạo được panorama cho cả bốn cặp, nhưng chất lượng không đều. Site1 còn cạnh nhòe hoặc kép trong vùng cắt, site3 có viền mờ ở tiền cảnh, site2 có ảnh kép rõ ở các vật gần máy; cặp khó giữ được bố cục cảnh dù còn nhiễu ảnh đêm và đường chuyển sáng. Tường và cửa sổ phía xa khớp trong khi vật gần máy bị lệch, đúng với ảnh hưởng của parallax lên một homography toàn cục. Kết quả synthetic ở mục 5.1 cho thấy pipeline ước lượng đúng hình học khi cảnh thỏa giả thiết; nó không loại trừ sai số của H trên cảnh thật.

Hai giới hạn của phương pháp hiện tại thể hiện rõ trong kết quả:

- Một homography toàn cục không mô tả được cảnh có nhiều độ sâu khi tâm máy dịch chuyển. Muốn giảm lỗi này phải chụp lại với tâm máy cố định hơn, hoặc dùng mô hình căn chỉnh cục bộ, nằm ngoài pipeline hiện tại.
- Trộn 50/50 không che ảnh kép và không làm mờ chênh sáng ở mép ảnh. Feather theo khoảng cách hoặc seam finding sẽ giảm đường chuyển sáng, nhưng đó là đổi phương pháp của S6.

Các phiên bản GPU giữ nguyên phương pháp nên sẽ có cùng các lỗi này. V1 được so với V0 dựa trên việc output giống nhau, không phải chất lượng tốt hơn.

<a name="muc-5-3"></a>
### 5.3 Thời gian từng stage

Benchmark chạy ba cặp chuẩn (site1, site2, site3) ở ba kích thước, tổng cộng chín cấu hình. Mỗi cấu hình có một lần warm-up không đo và 5 lần đo (`--runs 5`); bảng dùng trung bình của 5 lần đo. Thời gian lấy từ đồng hồ `std::chrono` trong chương trình C++, không dùng thời gian đo từ Python. Đọc, ghi ảnh và log debug nằm ngoài vùng đo. Cả chín cấu hình chạy trong cùng phiên Colab, cùng binary và cờ build. Script `scripts/run_benchmark.py` gọi `stitch` cho từng cấu hình và lưu nguyên output CSV; notebook đọc lại các CSV, kiểm tra đủ chín cấu hình và đủ số lần đo rồi dựng bảng.

Cặp khó không nằm trong benchmark: theo đề, ba cặp chuẩn dùng cho benchmark, còn cặp khó dùng để phân tích lỗi.

In [ ]:
average, minimum, measured, details = run_official_benchmark(RUNS)
stage_share = show_timing_tables(average, measured, details)

In [ ]:
plot_stage_share(average)
plot_stage_scaling(average)

<a name="muc-5-4"></a>
### 5.4 Hot stage theo độ phân giải

S4 là hot stage ở small và medium của cả ba site. Tỷ trọng trung bình của S4 giảm từ 56.2% ở small xuống 40.3% ở medium và 14.1% ở large. S4 so mọi cặp descriptor nên thời gian tính toán phụ thuộc trực tiếp vào số descriptor, còn độ phân giải ảnh ảnh hưởng gián tiếp thông qua số descriptor được tạo ra. Ở site1 và site2, cả hai ảnh đều đạt giới hạn 2000 descriptor ở mọi kích thước, nên S4 gần như đứng yên, từ 691.4 đến 702.4 ms; trên biểu đồ thứ hai, đường S4 nằm ngang. Site3 có ít descriptor hơn nên S4 chỉ từ 366.2 đến 521.7 ms, và lớn nhất ở medium, đúng kích thước mà site3 có nhiều descriptor nhất (1684 / 1780). Bản V0 hiện đếm bit bằng vòng lặp tuần tự, chưa gọi compiler builtin `__builtin_popcount`. Vì vậy, thời gian S4 còn chịu ảnh hưởng từ chi phí đếm bit. Có thể khảo sát tối ưu ở mức lệnh mà vẫn giữ thuật toán brute-force và kết quả Hamming; mức cải thiện cần được đo riêng, chưa được kiểm chứng trong bảng này.

Các stage xử lý theo pixel tăng gần tỷ lệ với kích thước ảnh. Từ small (0.92 Mpx) lên large (8.29 Mpx), S2 của site1 tăng từ 200.6 lên 1842.6 ms, S6 của site2 từ 207.6 lên 2056.3 ms. Vì vậy ở large, S2 đứng đầu ở site1 (1842.6 ms), còn S6 đứng đầu ở site2 (2056.3 ms) và site3 (1378.2 ms). S6 phụ thuộc diện tích canvas chứ không phải kích thước ảnh vào: canvas của site2 ở large rộng nhất (38.21 Mpx), nên S6 của site2 lớn nhất. Ở large, tỷ trọng trung bình của S2 là 32.2% và của S6 là 36.1%.

S1 và S3 tăng theo số pixel nhưng vẫn nhỏ: tỷ trọng trung bình của S1 đi từ 3.2% lên 7.1%, của S3 từ 4.4% lên 8.3%. S5 chỉ chiếm 3.9%, 2.2% và 2.2% ở ba kích thước, vì số match chỉ vài trăm và RANSAC dừng sớm; chưa nên tối ưu sâu S5 trước các stage đang chiếm phần lớn thời gian.

Hệ số biến thiên của Total qua 5 lần đo không quá 2.3% ở mọi cấu hình, nhỏ hơn nhiều so với chênh lệch giữa các stage, nên thứ hạng hot stage ở trên ổn định giữa các lần đo.

<a name="muc-5-5"></a>
### 5.5 Ước lượng theo định luật Amdahl

Nhóm dự kiến tăng tốc S2, S4 và S6 (mục 7). Với tập này, $p = (T_{S2} + T_{S4} + T_{S6}) / T_{\text{Total}}$, và speedup end-to-end không thể vượt $1/(1-p)$ dù ba stage này nhanh vô hạn, nếu phần còn lại giữ nguyên. Bảng tính $p$ cho từng site và kích thước từ thời gian trung bình ở mục 5.3. Các giá trị này dựa trên bản V0 hiện tại, trong đó S4 đếm bit bằng vòng lặp tuần tự và chưa gọi `__builtin_popcount`.

Nhóm không tính $p$ cho cả sáu stage, vì tổng sáu stage gần bằng Total nên $1/(1-p)$ gần như chia cho 0 và không cho giới hạn có ý nghĩa. Ở V1, chép dữ liệu giữa Host và Device cùng overhead launch kernel sẽ được đo riêng.

In [ ]:
amdahl = amdahl_table(average, ACCELERATED_STAGES)
plot_amdahl(amdahl)

Giới hạn speedup khi tăng tốc S2, S4, S6 là 6.34× đến 10.93× ở small, 6.07× đến 9.10× ở medium và 4.94× đến 6.31× ở large. Giới hạn giảm khi ảnh lớn lên vì S1 và S3, hai stage không nằm trong tập này, tăng theo số pixel và chiếm tỷ trọng lớn hơn. Site3 có $p$ thấp nhất ở cả ba kích thước, vì S4 của site3 nhỏ hơn (ít descriptor hơn) nên phần không được tăng tốc chiếm tỷ trọng lớn hơn.

Các giới hạn trên áp dụng khi chỉ tăng tốc S2, S4 và S6, còn các phần khác giữ nguyên; đây chưa phải speedup đã đo. V1 dự kiến đưa cả sáu stage lên GPU, nên cần đo lại toàn pipeline, gồm truyền dữ liệu và overhead. Giới hạn của kịch bản ba stage không phải trần speedup của V1. Với kịch bản chỉ tăng tốc ba stage này và phần còn lại giữ nguyên, giới hạn lớn nhất ở large trong lần đo hiện tại là 6.31×.

<a name="muc-6"></a>
## 6\. Dataset

### 6.1 Thu thập ảnh

Mỗi thành viên tự chụp ở địa điểm của mình theo cùng một quy định: đứng yên một chỗ và xoay máy giữa hai lần chụp, chồng lấn khoảng 30–50%, dùng camera chính 1x, không zoom, chụp hai ảnh liền nhau. Ảnh gốc được nộp vào thư mục chung trên Google Drive, đặt tên theo dạng `<loại cặp><số>_<left|right>.jpg`.

| Người chụp | Địa điểm | Cặp chính |
|---|---|---|
| Tuấn | Ẩn Coffee (quán cà phê, trong nhà và ngoài trời) | 2 chuẩn, 1 khó |
| Thành | Nhà thờ; đường ngoài trời; quán cà phê Houjicha (trong nhà); sau khuôn viên Trường Đại học Đồng Nai; trên đường đi | 4 chuẩn, 1 khó |
| Chương | Khuôn viên Trường Đại học Khoa học Tự nhiên, ĐHQG-HCM; khu ký túc xá | 1 chuẩn, 1 khó |
| Đế | Tiệm cà phê Mai An (trong nhà và ngoài trời) | 2 chuẩn, 1 khó |

### 6.2 Chọn cặp

Script `scripts/check_pairs.py` kiểm tra từng cặp chính theo các yêu cầu kiểm tra tự động được: tên file, định dạng JPEG, ảnh ngang, hai ảnh cùng kích thước và cùng thiết bị, chụp liền nhau, độ nét tương đối và ước lượng độ chồng lấn. Các tiêu chí script không kiểm tra được (cảnh tĩnh, ít parallax, đủ texture, không lộ mặt người lạ) được thành viên xem lại bằng mắt. Script xếp thứ tự đề xuất, sau đó nhóm chạy thử V0 trên tất cả cặp chính ở ba kích thước để xem cặp nào ra panorama. Ngày 18/10 cả nhóm chốt bộ dataset:

| Vai trò | Cặp | Địa điểm | Lý do chọn |
|---|---|---|---|
| site1, chuẩn | `thanh-standard1` | Nhà thờ, ngoài trời | Xếp đầu theo script, V0 ra panorama ở cả ba kích thước |
| site2, chuẩn | `tuan-standard2` | Ẩn Coffee, trong nhà | V0 ra panorama ở cả ba kích thước |
| site3, chuẩn | `de-standard2` | Tiệm cà phê Mai An, trong nhà | V0 ra panorama ở cả ba kích thước |
| Cặp khó | `chuong-challenge1` | Khu ký túc xá, ngoài trời, ban đêm | Cảnh đêm, chênh sáng mạnh, có người và xe chuyển động |

Ba cặp chuẩn ở ba địa điểm khác nhau, đúng yêu cầu của đề. Một số cặp không được chọn vì pipeline thất bại: `tuan-standard1` có mái pergola sát máy gây parallax, `thanh-standard3` có phép chiếu đưa một phần ảnh phải qua đường chân trời, còn `tuan-challenge1` có quá ít texture.

### 6.3 Chuẩn bị dataset

Script `scripts/prepare_dataset.py` xử lý mỗi cặp được chọn như sau:

1. Xoay ảnh theo thông tin EXIF orientation, vì stb đọc ảnh mà bỏ qua EXIF.
2. Cắt cả hai ảnh về tỷ lệ 16:9 bằng cùng một khung cắt ở giữa ảnh, không phóng to.
3. Thu nhỏ bằng Lanczos ra ba kích thước 1280×720 (small), 1920×1080 (medium), 3840×2160 (large).
4. Lưu JPEG chất lượng 95, bỏ EXIF (gồm cả tọa độ GPS), đặt tên dạng `dataset/site1/small_left.jpg`.
5. Ghi `dataset/manifest.csv` gồm ảnh nguồn, khung cắt và SHA-256 của từng ảnh.

Lệnh tạo lại `dataset/` từ ảnh gốc:

```bash
python scripts/prepare_dataset.py --raw raw --out dataset \
    --pair site1=thanh-standard1 --pair site2=tuan-standard2 \
    --pair site3=de-standard2 --pair challenge=chuong-challenge1
```

Bảng dưới đọc từ `dataset/manifest.csv` của phiên chạy; cell chuẩn bị ở mục 0 đã kiểm tra mọi ảnh đúng kích thước, là JPEG RGB, không còn EXIF và SHA-256 khớp manifest.

In [ ]:
show_dataset()

<a name="muc-7"></a>
## 7\. Kế hoạch chuyển sang GPU

### 7.1 Stage ưu tiên và căn cứ profiling

Theo mục 5.3 và 5.4, S2 (keypoint), S4 (ghép cặp) và S6 (warp và blend) chiếm phần lớn thời gian V0 ở cả ba độ phân giải. Nhóm ưu tiên ba stage này khi tối ưu ở V2 và V3:

- S4 là hot stage ở small và medium. Thời gian S4 tỷ lệ với $n_A \times n_B$ (tối đa 2000 × 2000 cặp descriptor); khi số descriptor giữ nguyên, thay đổi độ phân giải ảnh không trực tiếp làm tăng số cặp phải so. V0 hiện đếm bit bằng vòng lặp tuần tự, chưa gọi `__builtin_popcount`, nên chi phí tính Hamming cũng góp phần vào thời gian S4. Các cặp descriptor được so độc lập, phù hợp để xử lý song song trên GPU.
- S2 và S6 tăng theo số pixel và đứng đầu ở large. Cả hai chủ yếu là phép tính độc lập trên từng pixel.
- S1, S3 và S5 chiếm phần nhỏ hơn, nhưng phần của S1 và S3 tăng theo độ phân giải. Mục 5.5 cho thấy nếu chỉ tăng tốc S2, S4, S6 thì giới hạn speedup giảm khi ảnh lớn lên, nên ba stage còn lại cũng có kernel riêng để không trở thành phần giới hạn.

### 7.2 Mapping của V1

V1 dùng kernel đơn giản trên global memory và giữ đúng phương pháp, tham số và thứ tự tính của V0. Dữ liệu được giữ trên GPU giữa các stage để tránh chép qua lại; cách nối interface Host hiện tại của V0 với buffer trên Device sẽ được thống nhất khi cài V1.

| Stage | Một thread, warp hoặc block làm gì |
|---|---|
| S1 | Một thread cho một pixel ở mỗi lượt: đổi ảnh xám, blur ngang, blur dọc. Trọng số Gaussian trong constant memory |
| S2 | Một thread cho một pixel: segment test và score V; một thread cho một pixel: NMS 3×3, ghi ứng viên bằng bộ đếm atomic |
| S3 | Dùng lại kernel blur của S1 (σ = 2); một thread cho một keypoint tính 256 bit; pattern trong constant memory |
| S4 | Một block cho một descriptor A, các thread chia nhau duyệt descriptor B, tính Hamming bằng `__popc`, reduction trong block lấy hai láng giềng gần nhất |
| S5 | Một block cho một lần thử RANSAC: thread 0 sinh mẫu theo (seed, số thứ tự) và giải DLT, các thread chia nhau tính sai số, reduction theo đúng cây cộng của V0 |
| S6 | Một kernel một thread tính H⁻¹ và canvas; một thread cho một pixel canvas: chiếu về B, bilinear, trộn 50/50 với A |

Phần khó song song của từng stage:

- S2: chọn top-N cần sắp xếp toàn cục theo khóa (score, chỉ số quét); thứ tự output vẫn giống V0 vì khóa không trùng.
- S3: đọc pixel rải rác trong patch; loại keypoint ở biên phải giữ thứ tự bằng prefix sum.
- S4: ghi match theo thứ tự chỉ số A bằng prefix sum, không dùng atomic append.
- S5: điều kiện dừng phải xét kết quả theo thứ tự lần thử; SVD bằng `double` chạy tuần tự trong block.
- S6: Host phải đọc kích thước canvas để cấp phát output; phép tính `double`.

Thời gian mỗi stage trên GPU đo bằng CUDA events, gồm cả các bước phụ (sort, prefix sum, reduction, điều phối), không chỉ kernel chính. Thời gian chép dữ liệu giữa Host và Device ghi thành cột riêng.

### 7.3 Ý tưởng tối ưu dự kiến

Mỗi phiên bản sau V1 chỉ thêm một ý tưởng chính và đo riêng tác dụng của nó. Danh sách cuối chốt sau khi có profiling V1. Ba ý tưởng nhóm dự kiến thử trước, ứng với ba stage ưu tiên:

1. S4: dùng lại descriptor B qua shared memory. Một block xử lý nhiều descriptor A, nạp từng tile descriptor B vào shared memory một lần cho mọi A trong block, thay vì mỗi block đọc lại toàn bộ B từ global memory. Có thể thử thêm một warp cho một descriptor A, dùng warp shuffle để lấy hai láng giềng gần nhất.
2. S2: shared memory tile và gộp score với NMS. Mỗi block nạp tile ảnh cùng halo vào shared memory, tính score cho cả vùng halo rồi làm NMS ngay trong kernel, bỏ một lượt ghi và đọc lại bản đồ score. Thay sắp xếp toàn bộ ứng viên bằng chọn top-N theo histogram của score.
3. S6: ghi mỗi pixel canvas đúng một lần trong một kernel. Gộp xóa canvas, chép A và lấy mẫu B vào một kernel; chỉ chiếu và nội suy trong hình chữ nhật bao B; tính trước các số hạng của phép chiếu chỉ phụ thuộc hàng để giảm phép tính `double`.

Ở mức pipeline, nhóm cũng xem xét xử lý ảnh A và B trên hai CUDA stream ở S1 đến S3 và dùng pinned memory khi chép ảnh, nếu profiling cho thấy phần truyền dữ liệu đáng kể.

### 7.4 Rủi ro

- S5 và S6 dùng `double`. T4 có throughput phép tính 64 bit thấp hơn nhiều so với 32 bit, nên hai stage này có thể tăng tốc ít hơn dự kiến. Đổi sang `float` là đổi phương pháp, phải áp dụng cả cho V0.
- Giữ output giống V0: thứ tự keypoint của S2, thứ tự match và cách xử lý hai khoảng cách bằng nhau ở S4, điểm dừng RANSAC ở S5 đều phải giống hệt V0. Sort, atomic và reduction song song dễ làm đổi thứ tự nếu không có khóa xác định.
- Chi phí truyền dữ liệu: ở large, ảnh vào và nhất là panorama đều lớn, thời gian chép giữa Host và Device có thể vượt thời gian kernel. Nếu chỉ báo thời gian kernel, speedup end-to-end sẽ bị đánh giá cao hơn thực tế.
- S5 chỉ có vài trăm match, nên mỗi block có ít việc, còn phần giải DLT chạy tuần tự; S5 có thể không nhanh hơn V0 bao nhiêu.

### 7.5 Kiểm chứng V1

Mỗi stage của V1 được so với V0 trên cùng input: S2 và S3 so keypoint và descriptor từng bit, đúng thứ tự; S4 so danh sách match; S5 so điểm dừng, tập inlier và H qua sai số chiếu; S1 và S6 so ảnh với sai lệch cho phép đã ghi trong tài liệu thiết kế của stage. Sau đó chạy toàn pipeline trên bộ cặp synthetic với cùng ngưỡng sai số homography (RMSE ≤ 0.5 px, Max ≤ 1 px) và trên dataset ở cả ba độ phân giải.

<a name="muc-8"></a>
## 8\. Tài liệu tham khảo và khai báo dùng AI

### 8.1 Đề bài

- Đề đồ án CSC14120 năm 2026 (đề cuối kỳ) và đề nộp giữa kỳ: yêu cầu pipeline, dataset, cách đo thời gian, nội dung báo cáo.

### 8.2 Code bên ngoài

- Sean Barrett. [stb_image.h v2.30 và stb_image_write.h v1.16](https://github.com/nothings/stb), public domain hoặc MIT: đọc và ghi ảnh. Dùng nguyên, không sửa.

### 8.3 Nguồn cho từng stage

- [ITU-R Recommendation BT.601-7 (03/2011)](https://www.itu.int/rec/R-REC-BT.601). Dùng cho S1: hệ số đổi ảnh xám 0.299, 0.587, 0.114.
- [Rosten, E., Drummond, T. (2006). Machine learning for high-speed corner detection](https://www.edwardrosten.com/work/rosten_2006_machine.pdf), ECCV. Dùng cho S2: segment test, vòng tròn 16 điểm, score V (không dùng cây quyết định học máy).
- [Rosten, E., Porter, R., Drummond, T. (2010). Faster and better: a machine learning approach to corner detection](https://arxiv.org/abs/0810.2434), IEEE TPAMI. Dùng cho S2: so sánh độ lặp lại theo số điểm liên tiếp n.
- [OpenCV 4.x, `features2d.hpp`](https://github.com/opencv/opencv/blob/4.x/modules/features2d/include/opencv2/features2d.hpp) và [`orb.cpp`](https://github.com/opencv/opencv/blob/4.x/modules/features2d/src/orb.cpp). Dùng cho S2: ngưỡng FAST mặc định 10 (FAST) và 20 (ORB); S3: ORB làm mượt Gaussian σ = 2. Chỉ đọc để đối chiếu, không dùng OpenCV.
- [Calonder, M., Lepetit, V., Strecha, C., Fua, P. (2010). BRIEF: Binary Robust Independent Elementary Features](https://www.cs.ubc.ca/~lowe/525/papers/calonder_eccv10.pdf), ECCV. Dùng cho S3: test nhị phân, làm mượt σ = 2 trên cửa sổ 9×9, cách lấy mẫu G II.
- [Rublee, E., Rabaud, V., Konolige, K., Bradski, G. (2011). ORB: an efficient alternative to SIFT or SURF](https://doi.org/10.1109/ICCV.2011.6126544), ICCV. Dùng cho S2, S3: phương án xếp hạng Harris và rotated BRIEF, patch 31×31.
- [Vigna, S. SplitMix64 reference](https://prng.di.unimi.it/splitmix64.c). Dùng cho S3: sinh pattern BRIEF; S5: bộ sinh số ngẫu nhiên khi chọn mẫu.
- [Lowe, D. G. (2004). Distinctive Image Features from Scale-Invariant Keypoints](https://www.cs.ubc.ca/~lowe/papers/ijcv04.pdf), IJCV 60(2). Dùng cho S4: ý tưởng ratio test (mục 7.1–7.2).
- [OpenCV 4.13.0: Feature Matching](https://docs.opencv.org/4.13.0/dc/dc3/tutorial_py_matcher.html), [BFMatcher](https://docs.opencv.org/4.13.0/d3/da1/classcv_1_1BFMatcher.html), [Feature Matching with FLANN](https://docs.opencv.org/4.13.0/d5/d6f/tutorial_feature_flann_matcher.html), [lsh_index.h](https://github.com/opencv/opencv/blob/4.x/modules/flann/include/opencv2/flann/lsh_index.h). Dùng cho S4: Hamming, tìm kiếm vét cạn, ratio, cross-check và phương án tìm gần đúng (LSH). Chỉ đọc tài liệu, không dùng OpenCV.
- [Brown, M., Lowe, D. G. (2007). Automatic Panoramic Image Stitching using Invariant Features](https://www.cs.ubc.ca/~lowe/papers/07brown.pdf), IJCV 74. Dùng cho S5: RANSAC 4 điểm với DLT (mục 3.1); S6: so sánh trộn tuyến tính và multi-band (mục 6–7).
- [Harker, M., O'Leary, P. (2005). Computation of Homographies](https://www.robots.ox.ac.uk/~phst/BMVC2005/papers/190/paper190.pdf), BMVC. Dùng cho S5: chuẩn hóa tọa độ và DLT thuần nhất (mục 2.3).
- [Drmac, Z., Veselic, K. New Fast and Accurate Jacobi SVD Algorithm: II, LAPACK Working Note 170](https://www.netlib.org/lapack/lawnspdf/lawn170.pdf). Dùng cho S5: cơ sở của SVD Jacobi một phía và điều kiện dừng; nhóm tự cài cho 9 cột, không dùng LAPACK.
- [OpenCV 4.13.0: USAC](https://docs.opencv.org/4.13.0/de/d3e/tutorial_usac.html). Dùng cho S5: đối chiếu cách chọn mẫu, điểm MSAC, sai số chiếu và điều kiện dừng.
- [Fowler/Noll/Vo: FNV Hash](https://www.isthe.com/chongo/tech/comp/fnv/index.html). Dùng cho S5: hằng số của mã băm dùng để in dấu vết thứ tự mẫu trong log, không tham gia chọn H.
- [Chum, O., Matas, J. (2005). Matching with PROSAC – Progressive Sample Consensus](https://cmp.felk.cvut.cz/~matas/papers/chum-prosac-cvpr05.pdf), CVPR. Dùng cho S5: phương án chọn mẫu theo xếp hạng (không chọn).
- [Barath, D. et al. (2020). MAGSAC++, a Fast, Reliable and Accurate Robust Estimator](https://openaccess.thecvf.com/content_CVPR_2020/html/Barath_MAGSAC_a_Fast_Reliable_and_Accurate_Robust_Estimator_CVPR_2020_paper.html), CVPR. Dùng cho S5: phương án ước lượng có trọng số (không chọn).
- [Ivashechkin, M., Barath, D., Matas, J. (2021). VSAC: Efficient and Accurate Estimator for H and F](https://openaccess.thecvf.com/content/ICCV2021/html/Ivashechkin_VSAC_Efficient_and_Accurate_Estimator_for_H_and_F_ICCV_2021_paper.html), ICCV. Dùng cho S5: phương án RANSAC cải tiến (không chọn).
- [Szeliski, R. Image alignment and stitching: A tutorial](https://doi.org/10.1561/0600000009), Foundations and Trends in Computer Graphics and Vision 2(1). Dùng cho S6: hệ quy chiếu, lấy mẫu lại, trộn ảnh (mục 6).
- [Burt, P. J., Adelson, E. H. (1983). A multiresolution spline with application to image mosaics](https://doi.org/10.1145/245.247), ACM TOG 2(4). Dùng cho S6: nguồn gốc multi-band blending (phương án không chọn).

### 8.4 Các phương pháp ghép đặc trưng học sâu đã tham khảo

Khi so sánh phương án cho S4, nhóm đã tham khảo các phương pháp dưới đây nhưng không dùng phương pháp nào và không chạy thử trên dataset của nhóm.

- [Lindenberger, P., Sarlin, P.-E., Pollefeys, M. (2023). LightGlue: Local Feature Matching at Light Speed](https://openaccess.thecvf.com/content/ICCV2023/html/Lindenberger_LightGlue_Local_Feature_Matching_at_Light_Speed_ICCV_2023_paper.html), ICCV.
- [Wang, Y. et al. (2024). Efficient LoFTR: Semi-Dense Local Feature Matching with Sparse-Like Speed](https://openaccess.thecvf.com/content/CVPR2024/html/Wang_Efficient_LoFTR_Semi-Dense_Local_Feature_Matching_with_Sparse-Like_Speed_CVPR_2024_paper.html), CVPR.
- [Li, X., Rao, T., Pan, C. (2025). EDM: Efficient Deep Feature Matching](https://openaccess.thecvf.com/content/ICCV2025/html/Li_EDM_Efficient_Deep_Feature_Matching_ICCV_2025_paper.html), ICCV.
- [Edstedt, J. et al. RoMa v2: Harder Better Faster Denser Feature Matching](https://arxiv.org/abs/2511.15706), preprint 2025.
- [Lee, J., Kang, S., Yoo, S. (2026). MV-RoMa: From Pairwise Matching into Multi-View Track Reconstruction](https://arxiv.org/abs/2603.27542), preprint.

### 8.5 Tài liệu CUDA dùng cho kế hoạch GPU

- [NVIDIA CUDA C++ Best Practices Guide 13.0](https://docs.nvidia.com/cuda/archive/13.0.0/cuda-c-best-practices-guide/index.html). Dùng cho CUDA events, pinned memory, chồng truyền dữ liệu với tính toán, coalescing, shared memory, constant memory, kích thước block, throughput phép tính 64 bit trên compute capability 7.5, giới hạn Amdahl.
- [NVIDIA CUDA Math API 13.0: Integer Intrinsics](https://docs.nvidia.com/cuda/archive/13.0.0/cuda-math-api/cuda_math_api/group__CUDA__MATH__INTRINSIC__INT.html). Dùng cho S4: `__popc` trên word 32 bit.
- [NVIDIA CUDA Programming Guide: Warp Vote Functions](https://docs.nvidia.com/cuda/cuda-programming-guide/05-appendices/cpp-language-extensions.html#warp-vote-functions). Dùng cho S3: ghép bit descriptor bằng `__ballot_sync`.
- [NVIDIA CUDA C Programming Guide 12.5: Texture Fetching](https://docs.nvidia.com/cuda/archive/12.5.0/cuda-c-programming-guide/index.html#texture-fetching). Dùng cho S6: độ chính xác của bilinear phần cứng.

### 8.6 Khai báo dùng công cụ AI

Nhóm dùng ChatGPT, Codex và Claude Code để hỗ trợ giải thích thuật toán, gợi ý cách cài đặt, tìm lỗi, tổ chức notebook và rà soát câu chữ của báo cáo. Mọi đoạn code và số liệu đều được thành viên phụ trách chạy lại, kiểm chứng và phải giải thích được khi vấn đáp. Nguồn thuật toán và code bên ngoài được ghi ở chỗ sử dụng, trong các bảng trên và ở đầu các file nguồn liên quan.